# Environment Setup

In [1]:
import os
import sys

# 1. Define the "Check File"
# We create a dummy file to know if we've already fixed the environment this session.
DONE_FLAG = "/content/env_fixed.flag"

if not os.path.exists(DONE_FLAG):
    print("🔧 Fixing Environment (NumPy/PyArrow)...")

    # 2. Install uv (Instant)
    !pip install -q uv

    # 3. Force-Overwrite System Libraries (The only way to fix the Binary Conflict)
    #    'uv' is extremely fast. If files are already there, this takes < 2 seconds.
    !uv pip install --system --quiet \
        "pyarrow>=14.0.0,<15.0.0" \
        "numpy<2.0.0" \
        "pandas<2.2.0" \
        datasets huggingface_hub hf_transfer duckdb webdataset fastparquet Pillow faiss-cpu transformers gradio

    # 4. Mark as done
    !touch {DONE_FLAG}

    # 5. THE AUTOMATIC RESTART
    # This kills the kernel automatically so you don't have to click the button.
    print("🔄 Restarting Kernel to load new libraries... (Please wait a moment)")
    os.kill(os.getpid(), 9)

else:
    # After the restart, this block runs.
    import numpy as np
    import pyarrow
    import pandas as pd
    print("✅ Environment is Ready!")

✅ Environment is Ready!


In [2]:
# === Standard Library Imports ===
import csv
import gzip
import hashlib
import io
import gc
import json
import math
import os
import re
import shutil
import sqlite3
import subprocess
import shlex
import tarfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import islice
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union
from urllib.parse import urlparse

# === Third-Party Library Imports ===
import duckdb
import gdown
import glob
import numpy as np
import pandas as pd
import requests
import random
import torch
import webdataset as wds
import pyarrow as pa
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from huggingface_hub import HfApi, list_repo_files, login, list_repo_tree,hf_hub_url
from collections import Counter
from PIL import Image, UnidentifiedImageError
from requests.adapters import HTTPAdapter
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from urllib3.util import Retry

# === Google Colab Specific Imports ===
# (Only works in a Google Colab environment)
try:
    from google.colab import drive, userdata
except ImportError:
    print("Google Colab specific libraries not found. Skipping import.")

In [3]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Retrieve the token from Colab's Secrets Manager
hf_token = userdata.get('HF_TOKEN_2')

# Log in to HF using the retrieved token
login(hf_token)

In [4]:
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


# Working with Hugging Face Datasets

### Downloading files using HF datasets

In [ ]:
# Dwnloading the data from hugging face datasets.
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Clothing_Shoes_and_Jewelry", trust_remote_code=True)
items = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Clothing_Shoes_and_Jewelry", split="full", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
print(reviews["full"])
print(items["full"])

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 66033346
})

### Counting the Non-Null values in each column.

In [ ]:
def count_non_null(batch, columns):
    """Count non-null values in specified columns for a batch"""
    counts = {col: [] for col in columns}
    batch_size = len(batch[list(batch.keys())[0]]) # Get the size of the batch

    for i in range(batch_size):
        for col in columns:
            if col in batch:
                # Check if the value for the current example and column is not None
                # IN CASE OF PRICE COLUMN IT IS 'None' STRING and in case of features and descriptions it is '[]' empty list,
                # so do the appropriate changes accordingly in below code
                counts[col].append(1 if batch[col][i] is not None else 0)
            else:
                # If column not in batch, append 0 for this example
                counts[col].append(0)
    return counts

# Columns to check
columns = ['average_rating', 'rating_number']

# Count non-null values with multiprocessing
results = items.map(
    count_non_null,
    fn_kwargs={'columns': columns},
    batched=True,
    batch_size=1000,
    num_proc=4,  # Use 4 processes
    remove_columns=items.column_names  # Remove original columns to save memory
)

# Aggregate results
final_counts = {col: sum(results[col]) for col in columns}

# Print results
print("Non-null value counts:")
for col, count in final_counts.items():
    print(f"- {col}: {count:,} ({(count/len(items))*100:.1f}%)")

Map (num_proc=4):   0%|          | 0/7218481 [00:00<?, ? examples/s]

Non-null value counts:
- average_rating: 7,218,481.0 (100.0%)
- rating_number: 7,218,481 (100.0%)


In [ ]:
# NOT RECOMMENDED: This is a very bad way to check null values as the RAM will spike a lot.
column_name = 'helpful_vote'
x =reviews['full'][column_name]
print(x[:10])
add_ = 0
for i in x:
  if i == 0:
    add_ += 1
print(add_)

### Benchmarking the performance of HF datasets and DuckDB

In [ ]:
# To benchmark the difference in execution time of HF datasets and DuckDB!!!!
NPROC = min(8, os.cpu_count() or 2)

t0 = time.perf_counter()
# Filter keeps only verified rows (runs in parallel on CPU)
verified_ds = reviews.filter(lambda x: bool(x["verified_purchase"]), num_proc=NPROC)
count_ds = verified_ds.num_rows['full']
t1 = time.perf_counter()

print(f"[datasets] verified count = {count_ds:,}  | time = {t1 - t0:.2f}s  | num_proc={NPROC}")

In [ ]:
# TO use DuckDB we need to first convert the required data to parquet format for maximum efficiency.
parquet_dir = "/content/reviews_parquet"

# Keep only what we need for this quick benchmark to keep files tiny
need_cols = [c for c in reviews['full'].column_names if c in ("verified_purchase",)]
reviews_small = reviews['full'].remove_columns([c for c in reviews['full'].column_names if c not in need_cols])

# Write shards to Parquet (this creates multiple files under the folder)
reviews_small.to_parquet(parquet_dir)

In [ ]:
parquet_path = "/content/reviews_parquet"  # <-- use YOUR actual file path

con = duckdb.connect()
con.execute(f"PRAGMA threads={min(8, os.cpu_count() or 2)};")
con.execute("PRAGMA memory_limit='8GB';")

t0 = time.perf_counter()
count_duck = con.execute("""
  SELECT COUNT(*)
  FROM read_parquet(?)
  WHERE verified_purchase = TRUE
""", [parquet_path]).fetchone()[0]
t1 = time.perf_counter()

print(f"[duckdb ] verified count = {count_duck:,} | time = {t1 - t0:.2f}s")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[duckdb ] verified count = 62,175,766 | time = 2.16s


***CLEARLY DUCKDB IS BLAZINGLY FAST !!!***

### Preprocessing before converting to parquet.

In [ ]:
# REVIEWS — keep slim columns, filter verified
rev_keep = ["user_id","parent_asin","timestamp","rating","verified_purchase","helpful_vote"]
reviews_slim = reviews["full"].remove_columns([c for c in reviews["full"].column_names if c not in rev_keep])
reviews_slim = reviews_slim.filter(lambda x: bool(x["verified_purchase"]), num_proc=4)
# (Optional) drop the flag now that you’ve filtered:
reviews_slim = reviews_slim.remove_columns(["verified_purchase"])

Filter (num_proc=4):   0%|          | 0/66033346 [00:00<?, ? examples/s]

In [ ]:
# ITEMS — extract only what you need; we’ll map images→main_image_url later
itm_keep = ["parent_asin","main_category","title","average_rating","rating_number","price","images","categories","features","description","categories","details"]
items_slim = items.remove_columns([c for c in items.column_names if c not in itm_keep])

### Extracting the Main Image URL

In [ ]:
NPROC = min(8, os.cpu_count() or 2)

def _first_url(val):
    """Return the first non-empty string URL found inside val (str/list/ndarray/dict)."""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s or None
    if isinstance(val, (list, tuple, np.ndarray)):
        for x in val:
            u = _first_url(x)
            if u:
                return u
        return None
    if isinstance(val, dict):
        # common keys in the Amazon dumps; prioritize hi_res -> large -> medium -> url
        for k in ("hi_res", "large", "thumb"):
            if k in val:
                u = _first_url(val[k])
                if u:
                    return u
        return None
    # anything else
    return None

def to_main_url(batch):
    """datasets.map(batched=True) callback: reads batch['images'] (dicts) -> main_image_url"""
    out = []
    for img in batch["images"]:
        url = None
        if isinstance(img, dict):
            url = _first_url(img)           # dict case (your schema)
        else:
            url = _first_url(img)           # be tolerant to accidental list/str formats
        out.append(url)
    batch["main_image_url"] = out
    return batch


In [ ]:
# items_slim must contain the 'images' column
items_slim = items_slim.map(to_main_url, batched=True, num_proc=NPROC)
items_slim = items_slim.remove_columns(["images"])

In [ ]:
# Checking how many products there are without ant image.
len(items_slim.filter(lambda x: x["main_image_url"] is None, num_proc=NPROC))

### Converting to parquet format and saving locally and in google drive

In [ ]:
SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_path = f"{SAVE_DIR}/reviews_small_unpart"
itm_path = f"{SAVE_DIR}/items_small_unpart"

reviews_slim.to_parquet(rev_path)

size_bytes = os.path.getsize(rev_path)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
print("size:", human(size_bytes))

items_slim.to_parquet(itm_path)

Creating parquet from Arrow format:   0%|          | 0/62176 [00:00<?, ?ba/s]

4352307484

In [ ]:
# SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_HF_datasets'
# os.makedirs(SAVE_DIR, exist_ok=True)

# rev_g = f'{SAVE_DIR}/reviews_small_unpart'
# itm_g = f'{SAVE_DIR}/items_small_unpart'

# # write with a proper extension + compression
# reviews_slim.to_parquet(rev_g, engine='pyarrow', compression='snappy', index=False)
# items_slim.to_parquet(itm_g, engine='pyarrow', compression='snappy', index=False)

# # quick check
# import os
# print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
# print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

# Downloading the parquet file from Google Drive Link

In [ ]:
# Your shared links (file IDs extracted below)
REV_ID = "1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn"
ITM_ID = "155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq"

SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_OUT = f'{SAVE_DIR}/reviews_small_unpart'
ITM_OUT = f'{SAVE_DIR}/items_small_unpart'

gdown.download(id=REV_ID, output=REV_OUT, quiet=False)
gdown.download(id=ITM_ID, output=ITM_OUT, quiet=False)

# (optional) show sizes
def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print("reviews size:", human(os.path.getsize(REV_OUT)))
print("items   size:", human(os.path.getsize(ITM_OUT)))

Downloading...
From (original): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn
From (redirected): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn&confirm=t&uuid=f1dcb80b-1a8d-4b99-89ea-1d047a635e41
To: /content/artifacts/Post_HF_datasets/reviews_small_unpart
100%|██████████| 1.91G/1.91G [00:24<00:00, 78.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq
From (redirected): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq&confirm=t&uuid=3b9fdca4-bcc6-45d7-b5c1-038dbba32d51
To: /content/artifacts/Post_HF_datasets/items_small_unpart
100%|██████████| 4.20G/4.20G [00:46<00:00, 91.3MB/s]

reviews size: 1.78 GB
items   size: 3.91 GB


In [ ]:
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=REV_OUT)
items = load_dataset("parquet", data_files=ITM_OUT)

et = time.time()
print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 108.02244067192078 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Loading and Uploading data using HF

In [ ]:
api = HfApi()

repo_id = "PirateKing0402/Amazon_dataset"   # change this
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset')

### Uploading data to HF

In [ ]:
api.upload_file(
    path_or_fileobj=REV_OUT,
    path_in_repo="reviews_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add reviews parquet unpartitioned (hf_transfer)"
)

api.upload_file(
    path_or_fileobj=ITM_OUT,
    path_in_repo="items_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add items parquet unpartitioned (hf_transfer)"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/reviews_small_unpart         :   0%|          |  544kB / 1.91GB            

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/items_small_unpart           :   0%|          |  525kB / 4.20GB            

CommitInfo(commit_url='https://huggingface.co/datasets/PirateKing0402/Amazon_dataset/commit/a60281c9ad533dffaf04fc91a0b72f7ec7c29480', commit_message='Add items parquet unpartitioned (hf_transfer)', commit_description='', oid='a60281c9ad533dffaf04fc91a0b72f7ec7c29480', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset'), pr_revision=None, pr_num=None)

### Loading data from HF
( Slower than Google Drive download, better to use DuckDB to get data from HF )

In [ ]:
# Remove the reviews and items variables from memory
del reviews
del items

# You can optionally add print statements to confirm they are deleted (will raise NameError if successful)
# print(reviews)
# print(items)

In [ ]:
from datasets import load_dataset

# Define the repository ID and file paths within the repo
repo_id = "PirateKing0402/Amazon_dataset"
reviews_file_path = "reviews_small_unpart"
items_file_path = "items_small_unpart"
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{reviews_file_path}")
items = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{items_file_path}")
et = time.time()

print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

reviews_small_unpart:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

items_small_unpart:   0%|          | 0.00/4.20G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 969.474189043045 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Processing using DuckDB

### Using DuckDB on a data accessed through Remote Connection.
DuckDB's power lies in its ability to query massive remote datasets efficiently without local downloads. It achieves this by making many small, precise HTTP range requests to read only the parts of a file it needs.

This method is perfectly suited for cloud object stores like Amazon S3 or GCS, which are designed for this high-throughput access pattern. However, it triggers the anti-bot rate limits on standard web servers like the Hugging Face Hub, causing the query to fail.

In [ ]:
# ---- config: your repo + paths (file OR folder)
REPO_ID = "PirateKing0402/Amazon_dataset"
REV_PREFIX = "reviews_small_unpart"  # e.g. "reviews_small_unpart.parquet" OR a folder "reviews_small_unpart/"
ITM_PREFIX = "items_small_unpart"    # same idea for items

# Helper: build HTTPS URLs to the exact parquet files in the repo
def hf_parquet_urls(repo_id: str, prefix: str):
    files = list_repo_files(repo_id, repo_type="dataset")
    # case 1: a single file like "<prefix>.parquet"
    exact = [p for p in files if p == f"{prefix}"]
    if exact:
        return [f"https://huggingface.co/datasets/{repo_id}/resolve/main/{exact[0]}"]

rev_urls = hf_parquet_urls(REPO_ID, REV_PREFIX)
itm_urls = hf_parquet_urls(REPO_ID, ITM_PREFIX)

# Safety check: make sure we found files
print("review files:", len(rev_urls))
print("item files  :", len(itm_urls))
assert rev_urls, "No review parquet found in the repo/path you provided."
assert itm_urls, "No item parquet found in the repo/path you provided."

review files: 1
item files  : 1


In [ ]:
# ---- DuckDB setup (HTTP range reads; only needed bytes fetched)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='8GB';")  # optional

# ---- Items: count duplicate rows by parent_asin
sql_items = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup = con.execute(sql_items, {"urls": itm_urls}).fetchdf()

In [ ]:
sql_reviews = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup = con.execute(sql_reviews, {"urls": rev_urls}).fetchdf()

print("\nItems duplicates (pandas equivalence): duplicated(subset=['parent_asin']).sum()")
print(int(items_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup.loc[0, "keys_with_duplicates"]))

print("\nReviews duplicates (pandas equivalence): duplicated(subset=['user_id','parent_asin']).sum()")
print(int(reviews_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup.loc[0, "keys_with_duplicates"]))

### Using DuckDB on locally downloaded data.

#### Handling Duplicates

In [ ]:
# ---- DuckDB setup (using local files)
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

# Define local file paths
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

In [ ]:
# ---- Items: count duplicate rows by parent_asin using local file
sql_items_local = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup_local = con.execute(sql_items_local, [ITM_PATH_LOCAL]).fetchdf()

print("Items duplicates (local file):")
print(int(items_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup_local.loc[0, "keys_with_duplicates"]))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items duplicates (local file):
0 | keys_with_duplicates: 0


In [ ]:
# ---- Reviews: count duplicate rows by user_id and parent_asin using local file
sql_reviews_local = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup_local = con.execute(sql_reviews_local, [REV_PATH_LOCAL]).fetchdf()

print("\nReviews duplicates (local file):")
print(int(reviews_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup_local.loc[0, "keys_with_duplicates"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Reviews duplicates (local file):
803670 | keys_with_duplicates: 684012


In [ ]:
# Using DuckDB on locally downloaded data to count rows with missing timestamps
sql_reviews_null_timestamp = """
WITH t AS (
  SELECT TRY_CAST("timestamp" AS BIGINT) AS ts
  FROM read_parquet(?)
)
SELECT COUNT(*) FROM t WHERE ts IS NULL;
"""
reviews_null_timestamp_count = con.execute(sql_reviews_null_timestamp, [REV_PATH_LOCAL]).fetchone()[0]

print(f"Number of rows with null timestamp in reviews (local file): {reviews_null_timestamp_count}")

Number of rows with null timestamp in reviews (local file): 0


In [ ]:
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

SAVE_DIR = '/content/artifacts/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_DEDUP_LOCAL = f"{SAVE_DIR}/reviews_small_unpart"          # <-- output file to create
ITM_DEDUP_LOCAL = f"{SAVE_DIR}/items_small_unpart"

os.makedirs(os.path.dirname(REV_DEDUP_LOCAL), exist_ok=True)
os.makedirs(os.path.dirname(ITM_DEDUP_LOCAL), exist_ok=True)

# 0) how many rows before
orig_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_PATH_LOCAL]).fetchone()[0]
print("rows before:", orig_cnt)

rows before: 62175766


In [ ]:
# 1) write a deduplicated copy:
#    - partition by (user_id, parent_asin)
#    - order by timestamp DESC so we keep the newest
#    - tie-breakers: helpful_vote DESC, rating DESC (optional but sensible)
#    - keep exactly one row: rn = 1
def sql_quote(path: str) -> str:
    # escape any single quotes in a file path for SQL
    return path.replace("'", "''")

src = sql_quote(REV_PATH_LOCAL)
dst = sql_quote(REV_DEDUP_LOCAL)

sql = f"""
COPY (
  WITH raw AS (
    SELECT
      user_id,
      parent_asin,
      TRY_CAST("timestamp" AS BIGINT) AS ts,  -- keep safe quoting
      rating,
      helpful_vote
    FROM read_parquet('{src}')
  ),
  ranked AS (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY user_id, parent_asin
        ORDER BY ts DESC NULLS LAST, helpful_vote DESC NULLS LAST, rating DESC NULLS LAST
      ) AS rn
    FROM raw
  )
  SELECT user_id, parent_asin, ts AS "timestamp", rating, helpful_vote
  FROM ranked
  WHERE rn = 1
) TO '{dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql)

# 2) check after
dedup_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_DEDUP_LOCAL]).fetchone()[0]
print("rows after :", dedup_cnt)
print("removed    :", orig_cnt - dedup_cnt)

# Copy the items data from the local path to the destination path
# Although we did not perform deduplication on items data,
# we are copying it to the Post_dedup folder for consistency with the reviews data.
itm_src = sql_quote(ITM_PATH_LOCAL)
itm_dst = sql_quote(ITM_DEDUP_LOCAL)

sql_copy_items = f"""
COPY (
  SELECT * FROM read_parquet('{itm_src}')
) TO '{itm_dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_copy_items)

print(f"Items data copied from {ITM_PATH_LOCAL} to {ITM_DEDUP_LOCAL}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows after : 61372096
removed    : 803670


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items data copied from /content/artifacts/Post_HF_datasets/items_small_unpart to /content/artifacts/Post_dedup/items_small_unpart


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g = f'{SAVE_DIR}/reviews_small_unpart'
itm_g = f'{SAVE_DIR}/items_small_unpart'

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_DEDUP_LOCAL}'))
TO '{rev_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{ITM_DEDUP_LOCAL}'))
TO '{itm_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# quick check
import os
print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reviews: 2541.530979156494 MB
items  : 3953.9629278182983 MB


#### Pre-necessities

In [ ]:
# Define the file ID from the shared link
file_id1 = "1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS"
file_id2 = "1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p"

# Define the local path to save the downloaded file
SAVE_DIR = '/content/artifacts/Working'
os.makedirs(SAVE_DIR, exist_ok=True)
local_file_path1 = f'{SAVE_DIR}/reviews' # You can rename the file as needed
local_file_path2 = f'{SAVE_DIR}/items'

# Download the file
print(f"Downloading file with ID: {file_id1} to {local_file_path1}")
gdown.download(id=file_id1, output=local_file_path1, quiet=False)

# Download the file
print(f"Downloading file with ID: {file_id2} to {local_file_path2}")
gdown.download(id=file_id2, output=local_file_path2, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS
From (redirected): https://drive.google.com/uc?id=1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS&confirm=t&uuid=2ea9b8c0-cbc5-45b2-987b-0a0d74c257b6
To: /content/artifacts/Working/reviews
100%|██████████| 2.66G/2.66G [00:37<00:00, 71.1MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p
From (redirected): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p&confirm=t&uuid=e3e6d133-1ab3-4058-a820-86fcfeb100b5
To: /content/artifacts/Working/items
100%|██████████| 4.15G/4.15G [01:14<00:00, 55.5MB/s]


'/content/artifacts/Working/items'

In [ ]:
# Establish DuckDB connection
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

#### Extracting the Main Image URL

In [ ]:
# Execute the query and fetch all results into a Python list
initial_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{local_file_path2}')").fetchone()[0]
print(initial_count)

7218481


In [ ]:
# Filter items data to remove rows where main_image_url is null using DuckDB

ITM_FILTERED_LOCAL = f"{SAVE_DIR}/items_small_filtered"

# SQL query to filter and copy the data
sql_filter_items = f"""
COPY (
  SELECT *
  FROM read_parquet('{local_file_path2}')
  WHERE main_image_url IS NOT NULL
) TO '{ITM_FILTERED_LOCAL}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_filter_items)

# Quick check of the number of rows after filtering
filtered_items_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ITM_FILTERED_LOCAL}')").fetchone()[0]
print(f"Number of rows in items data after filtering for main_image_url: {filtered_items_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of rows in items data after filtering for main_image_url: 7208433


In [ ]:
temp4 = con.execute(f"SELECT main_image_url from read_parquet('{ITM_FILTERED_LOCAL}')").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
for item in temp4:
  # The URL is the first element of the tuple
  url = item[0] if isinstance(item, tuple) and len(item) > 0 else "no no no"
  if url and isinstance(url, str) and url.startswith('http'):
    pass
  else:
    print(f"Found a URL that does not start with 'http' or is not a valid URL: {url}")
    print(type(url))

print("Finished checking URLs.")

Finished checking URLs.


#### Extracting the gender column

In [ ]:
items_with_gender_clean = "/content/artifacts/Working/items_with_gender_clean"

os.makedirs(os.path.dirname(items_with_gender_clean), exist_ok=True)


sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{ITM_FILTERED_LOCAL}')
  ),
  x AS (
    SELECT
      *,
      regexp_extract(CAST(details AS VARCHAR),
                     '(?i)"department"\s*:\s*"([^"]+)"', 1) AS gender_raw
    FROM src
    WHERE regexp_matches(CAST(details AS VARCHAR),
                         '(?i)"department"\s*:\s*"')
  ),
  norm AS (
    SELECT
      *,
      lower(
        regexp_replace(
          regexp_replace(gender_raw, '[^A-Za-z -]+', ''),   -- keep letters / space / hyphen
          '\s+', ' '                                       -- collapse whitespace
        )
      ) AS dep_norm
    FROM x
  ),
  map AS (
    SELECT
      *,
      CASE
        -- specific buckets first
        WHEN dep_norm LIKE '%baby%girl%' OR dep_norm LIKE '%girl%baby%' THEN 'baby-girls'
        WHEN dep_norm LIKE '%baby%boy%'  OR dep_norm LIKE '%boy%baby%'  THEN 'baby-boys'
        WHEN dep_norm LIKE '%unisex%baby%'                               THEN 'unisex-baby'
        WHEN dep_norm LIKE '%unisex%child%'                              THEN 'unisex-child'
        WHEN dep_norm LIKE '%unisex%adult%'                              THEN 'unisex-adult'
        -- canonicalize women/men to womens/mens
        WHEN dep_norm LIKE '%womens%' OR dep_norm LIKE '%women%'         THEN 'womens'
        WHEN dep_norm LIKE '%mens%'   OR dep_norm LIKE '%men%'           THEN 'mens'
        WHEN dep_norm LIKE '%girls%'  OR dep_norm LIKE '%girl%'          THEN 'girls'
        WHEN dep_norm LIKE '%boys%'   OR dep_norm LIKE '%boy%'           THEN 'boys'
        ELSE NULL
      END AS gender
    FROM norm
  ),
  kept AS (
    SELECT *
    FROM map
    WHERE gender IN (
      'womens','mens','girls','boys',
      'baby-girls','baby-boys',
      'unisex-adult','unisex-child','unisex-baby'
    )
  )
  SELECT * EXCLUDE (dep_norm, gender_raw) FROM kept
) TO '{items_with_gender_clean}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_gender_clean)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_gender_clean


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_gender_clean}')").fetchone()[0]

6271634

#### Extracting the dimensions and weights

In [ ]:
items_with_dimension_weight = "/content/artifacts/Working/items_with_dimension_weight"
os.makedirs(os.path.dirname(items_with_dimension_weight), exist_ok=True)

con = duckdb.connect()

# 0) Source view with a text copy
con.execute(f"""
CREATE OR REPLACE TEMP VIEW src AS
SELECT *, CAST(details AS VARCHAR) AS dtxt
FROM read_parquet('{items_with_gender_clean}');
""")
print("src count:", con.execute("SELECT COUNT(*) FROM src").fetchone()[0])

src count: 6271634


In [ ]:
# 1) Create the view (no fetch here)
con.execute(r"""
CREATE OR REPLACE TEMP VIEW dims5 AS
SELECT
  *,
  -- 1) Package Dimensions (we can keep it simple if we also extract item-package separately)
  regexp_extract(dtxt, '(?is)"package\s+dimensions"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_package,

  -- 2) Product Dimensions
  regexp_extract(dtxt, '(?is)"product\s+dimensions"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_product,

  -- 3) Item Package Dimensions L x W x H
  regexp_extract(dtxt, '(?is)"item\s+package\s+dimensions\s*l\s*x\s*w\s*x\s*h"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_item_pkg_lxwxh,

  -- 4 & 5) Item Dimensions LxWxH (handles one or many spaces before LxWxH)
  regexp_extract(dtxt, '(?is)"item\s+dimensions\s+lxwxh"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_item_lxwxh
FROM src;
""")



In [ ]:
con.execute(r"""
CREATE OR REPLACE TEMP VIEW prio AS
SELECT
  *,
  CASE
    WHEN dim_product        IS NOT NULL AND dim_product != '' THEN dim_product
    WHEN dim_item_lxwxh     IS NOT NULL AND dim_item_lxwxh != '' THEN dim_item_lxwxh
    WHEN dim_item_pkg_lxwxh IS NOT NULL AND dim_item_pkg_lxwxh != '' THEN dim_item_pkg_lxwxh
    WHEN dim_package        IS NOT NULL AND dim_package != '' THEN dim_package
    ELSE NULL
  END AS dimensions_raw
FROM dims5;
""")

In [ ]:
# Add weight (Item Weight → Package Weight), then build dimension_weight with CASE
con.execute(r"""
CREATE OR REPLACE TEMP VIEW with_weight AS
SELECT
  p.*,
  CASE
    WHEN regexp_matches(dtxt, '(?is)"item weight"[^:：]*[:：]\s*["“]')
      THEN regexp_extract(dtxt, '(?is)"item weight"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1)
    WHEN regexp_matches(dtxt, '(?is)"package weight"[^:：]*[:：]\s*["“]')
      THEN regexp_extract(dtxt, '(?is)"package weight"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1)
    ELSE NULL
  END AS weight_raw
FROM prio p;
""")

In [ ]:
con.execute(r"""
CREATE OR REPLACE TEMP VIEW final AS
SELECT
  * EXCLUDE (dtxt, dim_package, dim_product, dim_item_pkg_lxwxh, dim_item_lxwxh, dimensions_raw, weight_raw),
  CASE
    WHEN dimensions_raw IS NOT NULL AND weight_raw IS NOT NULL
      THEN dimensions_raw || '; ' || weight_raw
    WHEN dimensions_raw IS NOT NULL
      THEN dimensions_raw
    ELSE weight_raw
  END AS dimension_weight
FROM with_weight where dimension_weight is not null;
""")

con.execute(f"COPY (SELECT * FROM final) TO '{items_with_dimension_weight}' (FORMAT PARQUET, CODEC ZSTD)")
print("Wrote:", items_with_dimension_weight)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_dimension_weight


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_dimension_weight}')").fetchone()[0]

5114268

#### Extracting the Manufacturer/Brand

In [ ]:
items_with_manufacturer = "/content/artifacts/Working/items_with_manufacturer"

os.makedirs(os.path.dirname(items_with_manufacturer), exist_ok=True)

sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_dimension_weight}')
  ),
  m AS (
    SELECT
      *,
      -- ONLY Manufacturer; tolerant to spacing, unicode colon, curly quotes
      NULLIF(
        trim(
          regexp_replace(
            regexp_extract(
              CAST(details AS VARCHAR),
              '(?is)"manufacturer"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1
            ),
            '\s+', ' '  -- collapse whitespace
          )
        ),
        ''
      ) AS manufacturer
    FROM src
  )
  SELECT *
  FROM m where manufacturer != ''
) TO '{items_with_manufacturer}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_manufacturer)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_manufacturer


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_manufacturer}')").fetchone()[0]

2683344

#### Extracting Main Category

In [ ]:
items_with_main_categories = "/content/artifacts/Working/items_with_main_categories"

os.makedirs(os.path.dirname(items_with_main_categories), exist_ok=True)

sql = f"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_manufacturer}')
  )
  SELECT
    * EXCLUDE(main_category),
    lower(
      list_element(
        list_filter(categories, x -> lower(x) IN ('clothing','shoes','jewelry')),
        1
      )
    ) AS main_categories
  FROM src where main_categories is not null
) TO '{items_with_main_categories}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_main_categories)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_main_categories


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_main_categories}')").fetchone()[0]

2045764

#### Extracting DFA (Date First Available)

In [ ]:
items_with_DFA = "/content/artifacts/Working/items_with_dfa"

os.makedirs(os.path.dirname(items_with_DFA), exist_ok=True)

sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_main_categories}')
  ),
  dfa AS (
    SELECT
      *,
      -- ONLY "Date First Available"; tolerant to spacing, unicode colon, curly quotes
      NULLIF(
        trim(
          regexp_replace(
            regexp_extract(
              CAST(details AS VARCHAR),
              '(?is)"date\s+first\s+available"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1
            ),
            '\s+', ' '  -- collapse whitespace
          )
        ),
        ''
      ) AS date_first_available
    FROM src where date_first_available != ''
  )
  -- keep schema, add the new column (you can drop 'details' if you don't need it)
  SELECT *
  FROM dfa
) TO '{items_with_DFA}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_DFA)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_dfa


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_DFA}')").fetchone()[0]

2032790

#### Extracting leaf category

In [ ]:
items_with_leaf_category = "/content/artifacts/Working/items_with_leaf_category"

os.makedirs(os.path.dirname(items_with_leaf_category), exist_ok=True) # Corrected variable name

sql = f"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_DFA}')
  )
  SELECT
    *,
    CASE
      WHEN categories IS NULL OR len(categories) = 0  -- Corrected function name
        THEN NULL
      ELSE
        lower(
          trim(
            list_element(categories, len(categories))  -- Corrected function name
          )
        )
    END AS leaf_category
  FROM src
) TO '{items_with_leaf_category}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_leaf_category)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_leaf_category


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_items_cleanup'
os.makedirs(SAVE_DIR, exist_ok=True)

itm_g = f'{SAVE_DIR}/items_cleaned'

sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{items_with_leaf_category}'))
TO '{itm_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# quick check
import os
print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

items  : 1216.1520309448242 MB


#### Extracting reviews based on filtered items

In [ ]:
REVIEWS_OUT = "/content/artifacts/Working/reviews_filtered_by_items"

os.makedirs(os.path.dirname(REVIEWS_OUT), exist_ok=True)

# Source views
con.execute(f"CREATE OR REPLACE TEMP VIEW items_ok AS SELECT parent_asin FROM read_parquet('{items_with_leaf_category}');")
con.execute(f"CREATE OR REPLACE TEMP VIEW reviews_raw AS SELECT * FROM read_parquet('{local_file_path1}');")

# Filter with a simple INNER JOIN (you said items have no duplicates)
con.execute("""
CREATE OR REPLACE TEMP VIEW reviews_filtered AS
SELECT r.*
FROM reviews_raw r
JOIN items_ok i
  ON i.parent_asin = r.parent_asin;
""")

# Counts (before/after) using the filtered relation you asked for
n_before = con.execute("SELECT COUNT(*) FROM reviews_raw").fetchone()[0]
n_after  = con.execute("SELECT COUNT(*) FROM reviews_filtered").fetchone()[0]
print(f"Reviews before: {n_before:,} | after filter: {n_after:,} | dropped: {n_before - n_after:,}")

# Persist filtered reviews
con.execute(f"""
COPY (SELECT * FROM reviews_filtered)
TO '{REVIEWS_OUT}' (FORMAT PARQUET, CODEC ZSTD);
""")
print("Wrote:", REVIEWS_OUT)

# Optional peek
df4 = con.execute("SELECT * FROM reviews_filtered LIMIT 100").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reviews before: 61,372,096 | after filter: 14,230,546 | dropped: 47,141,550


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/reviews_filtered_by_items


In [ ]:
df4.head()

,user_id,parent_asin,timestamp,rating,helpful_vote
0,AE226DXXSDWPBFTQB3M4VMOVZR2A,B07XFXXZMV,1631309290873,5.0,0
1,AE22ATMNSAFC5TQQ36NZZB2EYTVQ,B07FKVFNHC,1676746531768,5.0,0
2,AE22C2TILGPY7U23BJX2ZW4SVPMQ,B07VNRDVK3,1481071911000,5.0,0
3,AE22F4QOZJQ4N3ZUG7ZZUEOWHIJQ,B08DX4TJLQ,1647118774918,4.0,0
4,AE22GWOMEVMBAD7OWMX2KZQW756Q,B06XR38N19,1618352294318,4.0,0


#### Temporal Splitting

In [ ]:
TimestampLike = Union[pd.Timestamp, str, int, float]

def temporal_split_ms_duckdb(
    src_parquet: str,
    time_col: str = "ts",
    *,
    test_fraction: Optional[float] = None,
    cutoff: Optional[TimestampLike] = None,
    train_includes_cutoff: bool = True,
    drop_na_time: bool = True,
    sort_within_splits: bool = False,
    out_prefix: Optional[str] = "splits/v1"
) -> Tuple[int, pd.Timestamp, int, int]:
    if (test_fraction is None) == (cutoff is None):
        raise ValueError("Provide exactly one of `test_fraction` or `cutoff`.")
    if test_fraction is not None:
        if not np.isfinite(test_fraction) or not (0.0 < float(test_fraction) < 1.0):
            raise ValueError("`test_fraction` must be finite and in (0,1).")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE TEMP VIEW _raw AS SELECT * FROM read_parquet('{src_parquet}')")

    where_clause = ""
    if drop_na_time:
        where_clause = f"WHERE TRY_CAST({time_col} AS BIGINT) IS NOT NULL"

    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW _clean AS
        SELECT *, TRY_CAST({time_col} AS BIGINT) AS ms
        FROM _raw
        {where_clause}
    """)
    mn, mx, n = con.execute("SELECT MIN(ms), MAX(ms), COUNT(*) FROM _clean").fetchone()
    if n == 0:
        raise ValueError("All timestamps are NaN after parsing; nothing to split.")

    if cutoff is None:
        q = 1.0 - float(test_fraction)
        cutoff_ms = int(con.execute("SELECT quantile_disc(ms, ?) FROM _clean", [q]).fetchone()[0])
    else:
        if isinstance(cutoff, (int, float)) and np.isfinite(cutoff):
            cutoff_ms = int(cutoff)
        else:
            cutoff_ms = int(pd.to_datetime(cutoff, utc=True).value // 1_000_000)

    if not (mn <= cutoff_ms <= mx):
        raise RuntimeError(f"Cutoff {cutoff_ms} outside data range [{mn}, {mx}].")

    op_train = "<=" if train_includes_cutoff else "<"
    op_test  = ">"  if train_includes_cutoff else ">="

    con.execute(f"CREATE OR REPLACE TEMP VIEW train AS SELECT * FROM _clean WHERE ms {op_train} {cutoff_ms}")
    con.execute(f"CREATE OR REPLACE TEMP VIEW test  AS SELECT * FROM _clean WHERE ms {op_test}  {cutoff_ms}")

    train_n = con.execute("SELECT COUNT(*) FROM train").fetchone()[0]
    test_n  = con.execute("SELECT COUNT(*) FROM test").fetchone()[0]
    if train_n == 0 or test_n == 0:
        raise RuntimeError(f"Empty split: train={train_n}, test={test_n}. Adjust `test_fraction`/`cutoff`.")

    if out_prefix:
        train_query = "SELECT * FROM train ORDER BY ms" if sort_within_splits else "SELECT * FROM train"
        test_query  = "SELECT * FROM test  ORDER BY ms" if sort_within_splits else "SELECT * FROM test"

        con.execute(f"COPY ({train_query}) TO '{out_prefix}/reviews_small_train' (FORMAT PARQUET)")
        con.execute(f"COPY ({test_query})  TO '{out_prefix}/reviews_small_test'  (FORMAT PARQUET)")


    cutoff_ts = pd.to_datetime(cutoff_ms, unit="ms", utc=True).tz_convert(None)
    return cutoff_ms, cutoff_ts, train_n, test_n

In [ ]:
# Create the output directory if it doesn't exist
output_dir = '/content/artifacts/Working/Post_temporal_splits'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

cutoff_ms, cutoff_ts, n_tr, n_te = temporal_split_ms_duckdb(
    '/content/artifacts/Working/reviews_filtered_by_items',
    time_col='timestamp',
    test_fraction=0.20,
    # cutoff = "01-01-2022",
    train_includes_cutoff=True,
    sort_within_splits=True,
    out_prefix=output_dir
)

print(f"Temporal split complete. Cutoff date: {cutoff_ts}")
print(f"Train split rows: {n_tr:,}")
print(f"Test split rows: {n_te:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporal split complete. Cutoff date: 2021-06-25 11:52:13.539000
Train split rows: 11,384,437
Test split rows: 2,846,109


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_temporal_splits'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train'
rev_g_test = f'{SAVE_DIR}/reviews_small_test'

REV_SPLIT_TRAIN = "/content/artifacts/Working/Post_temporal_splits/reviews_small_train"
REV_SPLIT_TEST = "/content/artifacts/Working/Post_temporal_splits/reviews_small_test"

sql_copy_reviews_train = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_train)

sql_copy_reviews_test = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TEST}'))
TO '{rev_g_test}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_test)

# quick check
import os
print('reviews_train:', os.path.getsize(rev_g_train)/1024**2, 'MB')
print('items_test  :', os.path.getsize(rev_g_test)/1024**2, 'MB')

reviews_train: 492.9410934448242 MB
items_test  : 124.59007453918457 MB


#### Iterative k-core filtering

In [ ]:
def kcore_filter_iterative_duckdb(
    src_parquet: str,
    *,
    user_col: str = "user_id",
    item_col: str = "parent_asin",
    user_k: int = 5,
    item_k: int = 5,
    max_iters: int = 100,
    out_path: Optional[str] = "/content/splits/kcore_train",
    return_history: bool = True,
    return_df: bool = False,
    verbose: bool = False,
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    def q(ident: str) -> str:
        if '"' in ident:
            raise ValueError(f'Identifier {ident!r} contains a double quote (").')
        return f'"{ident}"'
    u, i = q(user_col), q(item_col)

    con = duckdb.connect()
    con.execute("PRAGMA disable_progress_bar")
    src = src_parquet.rstrip('/')

    # Start with a TABLE (materialized), not a view
    con.execute(f"CREATE OR REPLACE TEMP TABLE cur AS SELECT * FROM read_parquet('{src}')")
    n0 = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]
    if n0 == 0:
        raise ValueError("No rows to process.")

    history: List[Dict] = []

    for it in range(1, max_iters + 1):
        if verbose: print(f"Iteration {it} started")
        n_before = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]

        # User prune
        con.execute("DROP TABLE IF EXISTS ucnt")
        con.execute(f"CREATE TEMP TABLE ucnt AS SELECT {u} AS u, COUNT(*) AS c FROM cur GROUP BY {u}")
        con.execute("DROP TABLE IF EXISTS cur_u")
        con.execute(f"""
            CREATE TEMP TABLE cur_u AS
            SELECT c.* FROM cur c
            JOIN ucnt u ON c.{user_col} = u.u
            WHERE u.c >= {user_k}
        """)
        n_u = con.execute("SELECT COUNT(*) FROM cur_u").fetchone()[0]
        if n_u == 0:
            raise RuntimeError(f"All rows pruned at user step (iter={it}). Lower user_k/item_k.")

        # Item prune
        con.execute("DROP TABLE IF EXISTS icnt")
        con.execute(f"CREATE TEMP TABLE icnt AS SELECT {i} AS v, COUNT(*) AS c FROM cur_u GROUP BY {i}")
        con.execute("DROP TABLE IF EXISTS nxt")
        con.execute(f"""
            CREATE TEMP TABLE nxt AS
            SELECT u.* FROM cur_u u
            JOIN icnt v ON u.{item_col} = v.v
            WHERE v.c >= {item_k}
        """)
        n_after = con.execute("SELECT COUNT(*) FROM nxt").fetchone()[0]
        if n_after == 0:
            raise RuntimeError(f"All rows pruned at item step (iter={it}). Lower user_k/item_k.")

        users_after, items_after = con.execute(f"""
            SELECT COUNT(DISTINCT {u}), COUNT(DISTINCT {i}) FROM nxt
        """).fetchone()

        history.append({
            "iter": it,
            "rows_before": n_before,
            "rows_after": n_after,
            "users_after": users_after,
            "items_after": items_after,
            "removed": n_before - n_after,
        })

        # Convergence: no change this iteration
        if n_after == n_before:
            con.execute("DROP TABLE IF EXISTS cur")
            con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")
            if verbose: print(f"Iteration {it} finished (converged).")
            break

        # Prepare next iteration: replace cur with nxt (tables, so no cycles)
        con.execute("DROP TABLE IF EXISTS cur")
        con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")

    # Persist and/or return
    if out_path:
        con.execute(f"COPY (SELECT * FROM cur) TO '{out_path}' (FORMAT PARQUET)")
    out_df = con.execute("SELECT * FROM cur").df() if return_df else None
    hist_df = pd.DataFrame(history) if return_history else None
    return out_df, hist_df

In [ ]:
SAVE_DIR = '/content/artifacts/Working/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)
user_k = 5
item_k = 5
out_path = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

filtered_df, history = kcore_filter_iterative_duckdb(
    src_parquet='/content/artifacts/Working/Post_temporal_splits/reviews_small_train',  # or a folder with *.parquet
    user_col='user_id',
    item_col='parent_asin',
    user_k=user_k,
    item_k=item_k,
    max_iters=20,            # your data is already deduped earlier
    out_path=out_path,
    return_history=True,
    verbose=True,
    return_df=False                   # avoid pulling huge data back to RAM
)

Iteration 1 started
Iteration 2 started
Iteration 3 started
Iteration 4 started
Iteration 5 started
Iteration 6 started
Iteration 7 started
Iteration 8 started
Iteration 9 started
Iteration 10 started
Iteration 11 started
Iteration 11 finished (converged).


In [ ]:
history

,iter,rows_before,rows_after,users_after,items_after,removed
0,1,11384437,1171447,264366,53267,10212990
1,2,1171447,638062,106183,30716,533385
2,3,638062,547968,86184,26631,90094
3,4,547968,523998,81190,25520,23970
4,5,523998,516822,79705,25199,7176
5,6,516822,514757,79280,25107,2065
6,7,514757,513985,79132,25062,772
7,8,513985,513689,79070,25050,296
8,9,513689,513601,79050,25048,88
9,10,513601,513597,79049,25048,4


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

REV_K_CORE_TRAIN = f"/content/artifacts/Working/Post_k_core_filtering/reviews_small_train_{user_k}_{item_k}"

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_K_CORE_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

# quick check
import os
print('reviews_after_k_core_filtering:', os.path.getsize(rev_g_train)/1024**2, 'MB')

reviews_after_k_core_filtering: 21.00367546081543 MB


# Content Based Recommender system

## Loading cleaned items metadata to download the actual images and store it

In [ ]:
SAVE_DIR_DRIVE = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_items_cleanup'
ITM_G = f'{SAVE_DIR_DRIVE}/items_cleaned'

SAVE_DIR_LOCAL = '/content/artifacts/Working'
ITM_LOCAL = f'{SAVE_DIR_LOCAL}/items_cleaned'
os.makedirs(SAVE_DIR_LOCAL, exist_ok=True)



In [ ]:
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

# Directly copy the data from Google Drive to the local directory using DuckDB
# This avoids loading the entire dataset into a pandas DataFrame
sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{ITM_G}'))
TO '{ITM_LOCAL}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# Check the size of the copied file
size_bytes = os.path.getsize(ITM_LOCAL)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print(f"Copied data to {ITM_LOCAL}. Size: {human(size_bytes)}")

# You can still load a small sample into a DataFrame for inspection if needed
# items_df = con.execute(f"SELECT * FROM read_parquet('{ITM_LOCAL}') LIMIT 10").fetchdf()
# print("\nSample of the copied data:")
# display(items_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Copied data to /content/artifacts/Working/items_cleaned. Size: 1.19 GB


## Downloading Images to Google Drive

#### Config

In [ ]:
# ───────────────────────── config ─────────────────────────
ITEMS_PARQUET   = ITM_LOCAL   # must have: parent_asin, main_image_url, main_categories, gender
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"

FINAL_TAR_ROOT        = f"{BASE_DIR}/images_tars"         # where .tar shards go
TMP_TAR_ROOT  = "/content/tars_tmp"                       # fast local
MANIFEST_PATH   = f"{BASE_DIR}/images_manifest.csv.gz"    # gzipped CSV

UA                = "Mozilla/5.0 (compatible; recsys-downloader/2.0)"
MAX_RETRIES       = 3
MAX_WORKERS       = 64             # Drive is happier with modest concurrency
MAX_PER_HOST      = 25             # domain throttling: concurrent requests per host
BATCH_ROWS        = 50_000         # DuckDB -> Arrow batch size
FUTURES_IN_FLIGHT = 1_000          # keep in-flight futures bounded

# tune these as you like
CONNECT_TIMEOUT = 3.0
READ_TIMEOUT    = 20.0
POOL_CONNS      = 64           # per scheme
POOL_MAXSIZE    = 128          # concurrent pooled connections per host
RETRY_TOTAL     = 3
RETRY_BACKOFF   = 0.3
RETRY_STATUSES  = (429, 500, 502, 503, 504)
HEADERS = {"User-Agent": UA}   # reuse your UA string

# per-tar limits (WebDataset style)
MAX_IMAGES_PER_TAR = 15_000
MAX_TAR_BYTES      = 4_000_000_000  # ~4GB safety cap

Path(FINAL_TAR_ROOT).mkdir(parents=True, exist_ok=True)

#### Final Optimized downloading of images to google drive

In [ ]:
# ───────────────────── shard mapping (no leaf category) ─────────────────────
def compute_shard(df):
    g  = df["gender"].astype(str)
    mc = df["main_categories"].astype(str)

    cond1 = g.isin(["womens","mens"])
    cond2 = g.isin(["girls","boys","unisex-adult","unisex-child"])
    cond3 = g.isin(["baby-girls","baby-boys","unisex-baby"])

    # Convert np.nan to string 'nan' to avoid DTypePromotionError
    shard = np.where(cond1, mc + "|" + g,
             np.where(cond2, g,
             np.where(cond3, "baby", "nan")))
    df["shard"] = shard
    return df[df["shard"] != "nan"].copy() # Filter out rows where shard is 'nan' and return a copy

In [ ]:
# ───────────────────── per-host concurrency control ─────────────────────
_host_semaphores = {}
_host_lock = threading.Lock()

def host_semaphore(host: str) -> threading.Semaphore:
    with _host_lock:
        sem = _host_semaphores.get(host)
        if sem is None:
            sem = threading.Semaphore(MAX_PER_HOST)
            _host_semaphores[host] = sem
        return sem


In [ ]:
class TarShardManager:
    def __init__(self, tmp_root, final_root, max_items, max_bytes, name_fmt="{shard}-{seq:05d}.tar"):
        self.tmp_root   = Path(tmp_root)
        self.final_root = Path(final_root)
        self.max_items  = int(max_items)
        self.max_bytes  = int(max_bytes)
        self.name_fmt   = name_fmt
        self.tmp_root.mkdir(parents=True, exist_ok=True)
        self.final_root.mkdir(parents=True, exist_ok=True)
        self.open = {}  # shard -> meta

    def _next_seq_for_shard(self, shard: str) -> int:
        final_dir = self.final_root / shard
        if not final_dir.exists():
            return 0
        seq_re = re.compile(rf'^{re.escape(shard)}-(\d+)\.tar$')
        max_seq = -1
        for p in final_dir.iterdir():
            if not p.is_file():
                continue
            m = seq_re.match(p.name)
            if m:
                try:
                    s = int(m.group(1))
                    if s > max_seq: max_seq = s
                except ValueError:
                    pass
        return max_seq + 1  # start at 0 if none

    def _open_next(self, shard: str):
        meta = self.open.get(shard)
        next_seq = self._next_seq_for_shard(shard) if meta is None else meta["seq"] + 1

        tmp_dir   = self.tmp_root   / shard
        final_dir = self.final_root / shard
        tmp_dir.mkdir(parents=True, exist_ok=True)
        final_dir.mkdir(parents=True, exist_ok=True)

        # pick a seq that doesn't collide in FINAL
        fname      = self.name_fmt.format(shard=shard, seq=next_seq)
        final_path = final_dir / fname
        while final_path.exists():
            next_seq += 1
            fname      = self.name_fmt.format(shard=shard, seq=next_seq)
            final_path = final_dir / fname

        tmp_path = tmp_dir / (fname + ".part")
        tf = tarfile.open(tmp_path, mode="w")
        self.open[shard] = {
            "tar": tf, "count": 0, "seq": next_seq, "bytes": 0,
            "tmp_path": tmp_path, "final_dir": final_dir
        }

    def _finalize_current(self, shard: str, reopen: bool):
        meta = self.open.get(shard)
        if not meta or meta["tar"] is None:
            return None
        meta["tar"].close()

        fname      = self.name_fmt.format(shard=shard, seq=meta["seq"])
        final_path = meta["final_dir"] / fname

        # last-resort guard: never overwrite if somehow present
        if final_path.exists():
            new_seq = meta["seq"] + 1
            while (meta["final_dir"] / self.name_fmt.format(shard=shard, seq=new_seq)).exists():
                new_seq += 1
            final_path = meta["final_dir"] / self.name_fmt.format(shard=shard, seq=new_seq)

        shutil.move(str(meta["tmp_path"]), str(final_path))
        if reopen:
            self._open_next(shard)
        else:
            self.open[shard] = {"tar": None, "count": 0, "seq": meta["seq"],
                                "bytes": 0, "tmp_path": None, "final_dir": meta["final_dir"]}
        return str(final_path)

    def add(self, shard: str, member_path: str, data_bytes: bytes) -> str:
        if shard not in self.open or self.open[shard]["tar"] is None:
            self._open_next(shard)
        meta = self.open[shard]
        if meta["count"] >= self.max_items or meta["bytes"] >= self.max_bytes:
            self._finalize_current(shard, reopen=True)
            meta = self.open[shard]

        info       = tarfile.TarInfo(name=member_path)
        info.size  = len(data_bytes)
        info.mtime = int(time.time())
        meta["tar"].addfile(info, io.BytesIO(data_bytes))
        meta["count"] += 1
        meta["bytes"] += len(data_bytes)

        final_path = meta["final_dir"] / self.name_fmt.format(shard=shard, seq=meta["seq"])
        return str(final_path)

    def close_all(self, delete_tmp: bool = True):
        for shard in list(self.open.keys()):
            self._finalize_current(shard, reopen=False)
        self.open.clear()
        if delete_tmp:
            shutil.rmtree(self.tmp_root, ignore_errors=True)

In [ ]:
# ---------- pooled, thread-local Session ----------
_THREAD_LOCAL = threading.local()

def get_session() -> requests.Session:
    s = getattr(_THREAD_LOCAL, "session", None)
    if s is None:
        s = requests.Session()
        retry = Retry(
            total=RETRY_TOTAL,
            backoff_factor=RETRY_BACKOFF,
            status_forcelist=RETRY_STATUSES,
            allowed_methods=frozenset(["GET", "HEAD"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            pool_connections=POOL_CONNS,
            pool_maxsize=POOL_MAXSIZE,
            max_retries=retry,
        )
        s.mount("http://",  adapter)
        s.mount("https://", adapter)
        _THREAD_LOCAL.session = s
    return s

In [ ]:
# ───────────────────── http fetch → jpeg bytes ─────────────────────
def fetch_to_jpeg(asin, url, shard):
    """Return tuple for manifest: (asin,url,shard,ok,http_status,error,attempts,data_bytes)"""
    attempts = 0
    status   = None
    errtxt   = ""
    data     = None
    host     = urlparse(url).netloc or "unknown"
    sem      = host_semaphore(host)
    sess = get_session()

    for attempts in range(1, MAX_RETRIES+1):
        try:
            with sem:  # cap concurrency per host
                r = sess.get(
                    url,
                    headers=HEADERS,
                    timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
                    stream=False,   # full read into memory
                )
            status = r.status_code
            if status == 200 and r.content:
                try:
                    with Image.open(io.BytesIO(r.content)) as im:
                        im = im.convert("RGB")
                        buf = io.BytesIO()
                        im.save(buf, format="JPEG", quality=90, optimize=True)
                        data = buf.getvalue()
                    errtxt = ""
                    break
                except UnidentifiedImageError as e:
                    errtxt = f"pil_decode:{e.__class__.__name__}"
                except Exception as e:
                    errtxt = f"pil_other:{e.__class__.__name__}"
            else:
                errtxt = f"http_status:{status}"
        except requests.exceptions.Timeout:
            errtxt = "http_timeout"
        except requests.exceptions.ConnectionError:
            errtxt = "http_conn"
        except Exception as e:
            errtxt = f"http_other:{e.__class__.__name__}"

        time.sleep(0.3 * attempts)  # simple backoff

    ok = data is not None
    if ok:
        errtxt = ""
    return asin, url, shard, ok, status or "", errtxt, attempts, data or b""

In [ ]:
# ───────────────────── manifest writer (streaming, resume-aware) ─────────────────────
class ManifestWriter:
    def __init__(self, path):
        self.path = path
        # append mode; create if missing
        exists = os.path.exists(self.path)
        self.fh = gzip.open(self.path, "at", newline="")
        self.w  = csv.writer(self.fh)
        if not exists or os.stat(self.path).st_size == 0:
            self.w.writerow([
                "parent_asin","url","shard","hash_prefix",
                "tar_path","tar_member","ok","http_status","error","attempts","bytes"
            ])
        self.count = 0

    def write(self, row):
        self.w.writerow(row)
        self.count += 1
        if self.count % 5000 == 0:
            self.fh.flush()

    def close(self):
        try:
            self.fh.flush()
            self.fh.close()
        except Exception:
            pass


In [ ]:
def repair_tmp(tmp_root: str, final_root: str):
    """Move any stale *.tar.part from tmp_root to final_root/<shard>/<fname>.tar."""
    tmp_root_p  = Path(tmp_root)
    final_root_p= Path(final_root)
    if not tmp_root_p.exists():
        return
    for part in tmp_root_p.rglob("*.tar.part"):
        try:
            shard = part.parent.name
            final_dir = final_root_p / shard
            final_dir.mkdir(parents=True, exist_ok=True)
            final_name = part.name.replace(".tar.part", ".tar")
            shutil.move(str(part), str(final_dir / final_name))
        except Exception:
            # best effort: skip corrupted/locked files
            pass

In [ ]:
"""
────────────────────────────────────────────────────────────────────────────
TODO: ARCHITECTURE REFACTOR - ATOMIC CONSISTENCY (ACID)
────────────────────────────────────────────────────────────────────────────

1. THE PROBLEM (Current "Salvage" Logic):
   - We currently rely on `repair_tmp` to save partial TARs after a crash.
   - However, the Manifest CSV often lags behind the actual download.
   - Result: We save the TAR, but the Manifest doesn't know about it.
   - Next Run: The script re-downloads the same images -> DUPLICATES & WASTED BANDWIDTH.

2. THE SOLUTION (Atomic Commit):
   - Remove `repair_tmp`. On crash, we DELETE partial files (accept the loss).
   - Buffer Manifest rows in RAM (Python List) during the batch.
   - ONLY flush rows to CSV inside `_finalize_current` AFTER the TAR is safely on Drive.
   - Logic: If it's not in the Manifest, it effectively doesn't exist.

3. CONFIG TUNING (Minimizing Crash Penalty):
   - To make "deleting partial files" acceptable, we reduce TAR size.
   - Target: 5,000 images per TAR (approx 750MB - 1GB).
   - Speed: At 30 imgs/sec, 5,000 images takes ~3 minutes.
   - Benefit: A crash only wastes 3 minutes of work. Highly efficient.

IMPLEMENTATION STEPS:
   [ ] Set MAX_IMAGES_PER_TAR = 5000
   [ ] Set MAX_TAR_BYTES = 2 * 10**9
   [ ] Update TarShardManager to buffer rows list
   [ ] Update _finalize_current to take `manifest_writer` as arg and flush rows
   [ ] Delete repair_tmp function
────────────────────────────────────────────────────────────────────────────
"""

In [ ]:
from logging import raiseExceptions
# ───────────────────── driver with resume ─────────────────────
def run():

    # If a manifest exists, build a temp view of successfully downloaded rows
    if os.path.exists(MANIFEST_PATH):
        con.execute(f"""
            CREATE OR REPLACE TEMP VIEW man_ok AS
            SELECT parent_asin, url
            FROM read_csv_auto('{MANIFEST_PATH}', union_by_name=True)
            WHERE ok = 1
        """)
        resume_predicate = """
          AND NOT EXISTS (
              SELECT 1 FROM man_ok m
              WHERE m.parent_asin = i.parent_asin
                AND m.url        = i.main_image_url
          )
        """
        print("Resume: existing manifest found; rows with ok=1 will be skipped.")
    else:
        resume_predicate = ""  # nothing to skip
        print("Fresh run: no manifest found; downloading all rows.")

    # 1) Compute total to download (AFTER resume filter)
    total_remaining = con.execute(f"""
        SELECT COUNT(*) FROM (
          SELECT 1
          FROM read_parquet('{ITEMS_PARQUET}') i
          WHERE i.parent_asin IS NOT NULL
            AND i.main_image_url IS NOT NULL
            AND i.main_categories IS NOT NULL
            AND i.gender IS NOT NULL
            {resume_predicate}
        ) t
    """).fetchone()[0]

    print(f"Total remaining to download: {total_remaining:,}")

    # Stream items to download (skipping those already ok==1)
    cur = con.execute(f"""
        SELECT i.parent_asin, i.main_image_url, i.main_categories, i.gender
        FROM read_parquet('{ITEMS_PARQUET}') i
        WHERE i.parent_asin IS NOT NULL
          AND i.main_image_url IS NOT NULL
          AND i.main_categories IS NOT NULL
          AND i.gender IS NOT NULL
          {resume_predicate}
    """)
    reader = cur.fetch_record_batch(rows_per_batch=BATCH_ROWS)

    tars = TarShardManager(
        tmp_root=TMP_TAR_ROOT,
        final_root=FINAL_TAR_ROOT,
        max_items=MAX_IMAGES_PER_TAR,
        max_bytes=MAX_TAR_BYTES
    )
    mani  = ManifestWriter(MANIFEST_PATH)

    processed = 0
    ok_count  = 0
    pbar = tqdm(total=total_remaining, unit="img", desc="Downloading → tars")

    try:

        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break  # newer Arrow: signals end via exception

            if batch is None or batch.num_rows == 0:
                break

            df = batch.to_pandas(types_mapper=None)
            df = compute_shard(df)

            # Create job tuples (asin, url, shard)
            jobs_iter = (
                (asin, url, shard)
                for asin, url, shard in df[["parent_asin","main_image_url","shard"]].itertuples(index=False, name=None)
            )

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                inflight = set()

                def submit_some(n):
                    for _ in range(n):
                        try:
                            a, u, s = next(jobs_iter)
                        except StopIteration:
                            return False
                        inflight.add(ex.submit(fetch_to_jpeg, a, u, s))
                    return True

                submit_some(min(FUTURES_IN_FLIGHT, MAX_WORKERS * 16))

                while inflight:
                    for fut in as_completed(inflight):
                        inflight.remove(fut)
                        asin, url, shard, ok, status, errtxt, attempts, data = fut.result()
                        processed += 1

                        if ok:
                            h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
                            prefix = h[:2]
                            member = f"{prefix}/{asin}_{h}.jpg"
                            tar_path = tars.add(shard, member, data)
                            mani.write([asin, url, shard, prefix, tar_path, member, 1, status, "", attempts, len(data)])
                            ok_count += 1
                        else:
                            mani.write([asin, url, shard, "", "", "", 0, status, errtxt, attempts, 0])

                        # top up queue
                        submit_some(1)

                        # Progress shows processed & ok
                        pbar.set_postfix_str(f"ok={ok_count:,}")
                        pbar.update(1)

    finally:
        pbar.close()
        mani.close()
        tars.close_all(delete_tmp=True)
        print("\nEven if the system might have crashed the remaining tars_temp file was exported to Google drive safely. If it's a successful run then Congratulations buddy!!!😎")

    print(f"\nDone. Processed={processed:,}, ok={ok_count:,}.")
    print(f"Manifest: {MANIFEST_PATH}")
    print(f"Tar shards root: {FINAL_TAR_ROOT}")

# repair_tmp(TMP_TAR_ROOT, FINAL_TAR_ROOT)
run()

Resume: existing manifest found; rows with ok=1 will be skipped.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total remaining to download: 98


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Even if the system might have crashed the remaining tars_temp file was exported to Google drive safely. If it's a successful run then Congratulations buddy!!!😎

Done. Processed=98, ok=0.
Manifest: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_manifest.csv.gz
Tar shards root: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars


#### Analyzing Manifest data

In [ ]:
import os

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

total_size = 0
for dirpath, dirnames, filenames in os.walk(FINAL_TAR_ROOT):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        if not os.path.islink(fp):
            total_size += os.path.getsize(fp)

print(f"Size of {FINAL_TAR_ROOT}: {human(total_size)}")

Size of /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars: 264.91 GB


In [ ]:
import pandas as pd
import gzip

# Define the path to the manifest file
MANIFEST_PATH = f"/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_manifest.csv.gz"

# Read the gzipped CSV file into a pandas DataFrame
try:
    with gzip.open(MANIFEST_PATH, 'rt') as f:
        manifest_df = pd.read_csv(f)

    # Display the first few rows of the DataFrame
    print("Manifest file content:")
    display(manifest_df.head())

except FileNotFoundError:
    print(f"Error: Manifest file not found at {MANIFEST_PATH}")
except Exception as e:
    print(f"An error occurred while reading the manifest file: {e}")

/tmp/ipython-input-3953762423.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  manifest_df = pd.read_csv(f)


Manifest file content:


,parent_asin,url,shard,hash_prefix,tar_path,tar_member,ok,http_status,error,attempts,bytes
0,B00M44BSAG,https://m.media-amazon.com/images/I/514qGTrvTv...,clothing|womens,a2,/content/drive/MyDrive/Product_Recommender_End...,a2/B00M44BSAG_a25dd49db5.jpg,1,200,NaN,1,30546
1,B00E1HICEY,https://m.media-amazon.com/images/I/81P6eVm5Gy...,clothing|mens,76,/content/drive/MyDrive/Product_Recommender_End...,76/B00E1HICEY_76da1adf8e.jpg,1,200,NaN,1,199763
2,B00J62LNCW,https://m.media-amazon.com/images/I/81615aGbaQ...,shoes|mens,5e,/content/drive/MyDrive/Product_Recommender_End...,5e/B00J62LNCW_5e1743c97a.jpg,1,200,NaN,1,195485
3,B096Z2SMWT,https://m.media-amazon.com/images/I/71OvgY2Bh4...,unisex-child,03,/content/drive/MyDrive/Product_Recommender_End...,03/B096Z2SMWT_03ac9c4e34.jpg,1,200,NaN,1,251707
4,B075NXTTJW,https://m.media-amazon.com/images/I/61K1RxY2rR...,shoes|womens,ec,/content/drive/MyDrive/Product_Recommender_End...,ec/B075NXTTJW_ecd037bc0f.jpg,1,200,NaN,1,104278


In [ ]:
manifest_df.shape

(2033032, 11)

In [ ]:
manifest_df[manifest_df.ok == 0].head()

,parent_asin,url,shard,hash_prefix,tar_path,tar_member,ok,http_status,error,attempts,bytes
13727,B0016GZQUG,https://m.media-amazon.com/images/I/61W0Me1DsK...,jewelry|womens,NaN,NaN,NaN,0,404,http_status:404,3,0
50643,B0062S0NSS,https://m.media-amazon.com/images/I/51X7PfarDA...,girls,NaN,NaN,NaN,0,404,http_status:404,3,0
60999,B000UWVMVE,https://m.media-amazon.com/images/I/71BRrUmBHS...,shoes|womens,NaN,NaN,NaN,0,404,http_status:404,3,0
63651,B000FBRTRQ,https://m.media-amazon.com/images/I/716z69ip72...,shoes|mens,NaN,NaN,NaN,0,404,http_status:404,3,0
74997,B09DLQ3YVB,https://m.media-amazon.com/images/I/81jPy1DsTy...,clothing|womens,NaN,NaN,NaN,0,404,http_status:404,3,0


In [ ]:
print(manifest_df[manifest_df.ok == 0]["url"])

13727      https://m.media-amazon.com/images/I/61W0Me1DsK...
50643      https://m.media-amazon.com/images/I/51X7PfarDA...
60999      https://m.media-amazon.com/images/I/71BRrUmBHS...
63651      https://m.media-amazon.com/images/I/716z69ip72...
74997      https://m.media-amazon.com/images/I/81jPy1DsTy...
                                 ...                        
2033027    https://m.media-amazon.com/images/I/61bY5pqHPj...
2033028    https://m.media-amazon.com/images/I/716z69ip72...
2033029    https://m.media-amazon.com/images/I/71UwsypjEH...
2033030    https://m.media-amazon.com/images/I/81DigD6nsD...
2033031    https://m.media-amazon.com/images/I/91fHeJVKuf...
Name: url, Length: 340, dtype: object


#### Analyzing the tar files generated

In [ ]:
#FInding total tar files generated
!find /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars -type f | wc -l

180


In [ ]:
def list_all_tars(root):
    return [str(p) for p in Path(root).rglob("*.tar")]

In [ ]:
# expected_counts: { tar_path -> {"count": int, "bytes": int} }
expected_counts = {}

with gzip.open(MANIFEST_PATH, "rt", newline="") as fh:
    r = csv.DictReader(fh)
    for row in r:
        if row.get("ok") != "1":
            continue
        tar_path = row.get("tar_path", "")
        if not tar_path:
            continue
        rec = expected_counts.setdefault(tar_path, {"count": 0, "bytes": 0})
        rec["count"] += 1
        # bytes column might be empty; guard it
        try:
            rec["bytes"] += int(row.get("bytes", "0"))
        except ValueError:
            pass

len(expected_counts)

180

In [ ]:
# Generated whether the total images as per manifest matches the original number of items we had
total_count = 0

for rec in expected_counts.values():
    total_count += rec["count"]

total_count

2032692

In [ ]:
# Checking whether the actual tar files has all the images downloaded without any issue in each shard
def is_img(name: str):
    n = name.lower()
    return n.endswith(".jpg") or n.endswith(".jpeg") or n.endswith(".png")

mismatches = []  # collect problems to review later

all_tars = list_all_tars(FINAL_TAR_ROOT)
for tar_fp in tqdm(all_tars, desc="Checking tars vs manifest (counts)"):
    exp = expected_counts.get(tar_fp, {"count": 0, "bytes": 0})
    exp_count, exp_bytes = exp["count"], exp["bytes"]

    try:
        with tarfile.open(tar_fp, mode="r") as tf:
            cnt = 0
            sz  = 0
            for m in tf:
                if m.isreg() and is_img(m.name):
                    cnt += 1
                    sz  += m.size
    except Exception as e:
        mismatches.append((tar_fp, f"TAR_OPEN_ERROR:{e}"))
        continue

    ok = (cnt == exp_count)
    if not ok:
        mismatches.append((tar_fp, f"COUNT_MISMATCH exp={exp_count} got={cnt}"))

    # bytes can differ slightly (JPEG recompression), treat as soft check
    if exp_bytes and abs(sz - exp_bytes) > max(1024, 0.05 * exp_bytes):
        mismatches.append((tar_fp, f"BYTES_DRIFT exp={exp_bytes} got={sz}"))

# Summary
print(f"Tars checked: {len(all_tars)}")
print(f"Problems found: {len(mismatches)}")
if mismatches[:10]:
    for m in mismatches[:10]:
        print(m)


Checking tars vs manifest (counts):   0%|          | 0/181 [00:00<?, ?it/s]

Tars checked: 181
Problems found: 2
('/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003.tar', 'TAR_OPEN_ERROR:unexpected end of data')
('/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003-fixed.tar', 'COUNT_MISMATCH exp=0 got=369')


In [ ]:
# Checking if a particular tar file is readable or not if any issues were recogonized in the above code.

tar_fp = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003-fixed.tar"

def tar_ok(path):
    cmd = f"tar -tf {shlex.quote(path)} >/dev/null"
    return subprocess.call(["bash","-lc",cmd]) == 0

print("Readable?", tar_ok(tar_fp))

Readable? True


In [ ]:
# In case there were any issue in any tar file then how much of the tar file is recoverable.
import subprocess, shlex
# --ignore-zeros lets tar list/extract until the first broken spot
cmd = f"tar --ignore-zeros -tf {shlex.quote(tar_fp)} | head"
print(subprocess.check_output(["bash","-lc",cmd]).decode("utf-8"))


bb/B07H3W4T8F_bb883ccec0.jpg
52/B098921NRB_52f868ccc6.jpg
7b/B07SDMCN8D_7b3198d53d.jpg
82/B088GVQ83Y_823fa3649d.jpg
75/B07BZVS9N3_750f29dd6c.jpg
4c/B08KH7YJ49_4c8d517b45.jpg
28/B003X27I02_2889130d3b.jpg
10/B07JZLLP9C_105e0e5e50.jpg
96/B01HMRP3VY_963fc5612f.jpg
98/B07CNDP73V_9881c7e036.jpg



#### Repairing any broken tar files

In [ ]:
# <<< EDIT THIS >>>
BROKEN_TAR = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003.tar"

SHARD_DIR   = str(Path(BROKEN_TAR).parent)
SHARD_NAME  = Path(SHARD_DIR).name
PREFIX_NAME = SHARD_NAME                              # filenames use the shard name as prefix
print("Shard dir:", SHARD_DIR, "| Shard:", SHARD_NAME, "| Prefix:", PREFIX_NAME)

Shard dir: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens | Shard: jewelry|womens | Prefix: jewelry|womens


In [ ]:
def load_expected_for_tar(manifest_path: str, tar_path: str):
    rows = []
    with gzip.open(manifest_path, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") != "1":
                continue
            if row.get("tar_path") == tar_path:
                # we only need asin, url, and the member path we used originally
                rows.append({
                    "asin": row["parent_asin"],
                    "url": row["url"],
                    "member": row["tar_member"],   # e.g. "ab/ASIN_hash.jpg"
                })
    return rows

expected_rows = load_expected_for_tar(MANIFEST_PATH, BROKEN_TAR)
print("Expected images in this tar:", len(expected_rows))
print("Sample members:", [expected_rows[i]["member"] for i in range(min(3, len(expected_rows)))])

Expected images in this tar: 369
Sample members: ['bb/B07H3W4T8F_bb883ccec0.jpg', '52/B098921NRB_52f868ccc6.jpg', '7b/B07SDMCN8D_7b3198d53d.jpg']


In [ ]:
# Small helpers
def verify_tar_ok(path: str) -> bool:
    cmd = f"tar -tf {shlex.quote(path)} >/dev/null"
    return subprocess.call(["bash","-lc",cmd]) == 0

def next_fixed_name(broken_path: str) -> str:
    p = Path(broken_path)
    return str(p.with_name(p.stem + "-fixed" + p.suffix))  # e.g., womens-00003-fixed.tar

# One shared session (keeps connections warm)
sess = requests.Session()
sess.headers.update({"User-Agent": "recsys-repair/0.1"})

def fetch_jpeg_rgb(url: str, timeout=(3.0, 20.0), retries: int = 3):
    """Return (ok, bytes_or_empty). Re-encodes as RGB JPEG to normalize."""
    for attempt in range(1, retries+1):
        try:
            r = sess.get(url, timeout=timeout, stream=False)
            if r.status_code == 200 and r.content:
                try:
                    with Image.open(io.BytesIO(r.content)) as im:
                        im = im.convert("RGB")
                        buf = io.BytesIO()
                        im.save(buf, format="JPEG", quality=90, optimize=True)
                        data = buf.getvalue()
                    return True, data
                except UnidentifiedImageError:
                    pass
        except Exception:
            pass
        time.sleep(0.2 * attempt)
    return False, b""

def repair_tar_simple(broken_tar: str):
    new_tar  = next_fixed_name(broken_tar)
    new_part = new_tar + ".part"

    ok, fail = 0, 0
    errors   = []

    # Write sequentially (low RAM). After each loop we drop buffers and GC.
    tf = tarfile.open(new_part, mode="w")
    try:
        for row in tqdm(expected_rows,total = len(expected_rows), desc="Rebuilding (simple)", unit="img"):
            ok1, data = fetch_jpeg_rgb(row["url"])
            if ok1 and data:
                info = tarfile.TarInfo(name=row["member"])
                info.size  = len(data)
                info.mtime = int(time.time())
                tf.addfile(info, io.BytesIO(data))
                ok += 1
            else:
                errors.append((row["member"], row["url"]))
                fail += 1
            # cleanup per-iteration
            data = None
            gc.collect()
    finally:
        tf.close()

    # Verify and promote
    if verify_tar_ok(new_part) and ok > 0:
        os.replace(new_part, new_tar)
        verified = True
    else:
        verified = False

    return {"new_tar": new_tar, "new_part": new_part, "ok": ok, "fail": fail, "errors": errors, "verified": verified}

result = repair_tar_simple(BROKEN_TAR)
result


Rebuilding (simple):   0%|          | 0/369 [00:00<?, ?img/s]

{'new_tar': '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003-fixed.tar',
 'new_part': '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003-fixed.tar.part',
 'ok': 369,
 'fail': 0,
 'errors': [],
 'verified': True}

In [ ]:
def fast_count_images(path: str) -> int:
    out = subprocess.check_output(
        ["bash","-lc", f"tar -tf {shlex.quote(path)} | egrep -i '\\.(jpg|jpeg|png)$' | wc -l"]
    ).decode().strip()
    return int(out or "0")

if result["verified"]:
    old_bad = BROKEN_TAR + ".bad"
    if os.path.exists(old_bad):
        os.remove(old_bad)
    os.rename(BROKEN_TAR, old_bad)
    print("Moved old tar to:", old_bad)

    exp = len(expected_rows)
    got = fast_count_images(result["new_tar"])
    print(f"Expected: {exp}  |  New tar has: {got}  |  Missing: {exp-got}")

    if result["fail"]:
        print("Some URLs failed (first few):", result["errors"][:5])
else:
    print("Verification failed; leaving .part on disk for inspection:", result["new_part"])

Moved old tar to: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars/jewelry|womens/jewelry|womens-00003.tar.bad
Expected: 369  |  New tar has: 369  |  Missing: 0


## Loading data to Hugging Face

In [ ]:
api = HfApi()
repo_id = "PirateKing0402/Amazon-fashion-image-tars"
api.create_repo(repo_id, repo_type="dataset", private=False, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon-fashion-image-tars', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon-fashion-image-tars')

In [ ]:
all_tars = list(list_all_tars(FINAL_TAR_ROOT))

In [ ]:
src = Path(f"{BASE_DIR}/images_tars")
tmp = Path("/content/tmp/hf_up"); tmp.mkdir(parents=True, exist_ok=True)

# (optional but useful) skip files already on HF under "tars/"
print("Scanning remote repo to skip already-uploaded files (once per run)...")
remote_files = set()
# Assuming objects with a 'size' attribute are files
for it in list_repo_tree(repo_id, repo_type="dataset", recursive=True):
    # Check if the object has a 'size' attribute and its path starts with "tars/" and ends with ".tar"
    if hasattr(it, 'size') and it.path.startswith("tars/") and it.path.endswith(".tar"):
        remote_files.add(it.path)

to_upload = []
# Assuming 'src' variable is defined and is the path to the local tar files root
for p in all_tars:
    rel_path = f"tars/{p.relative_to(src).as_posix()}"
    if rel_path not in remote_files:
        to_upload.append(p)

print(f"Local tars found: {len(all_tars)} | Already on HF: {len(all_tars) - len(to_upload)} | To upload: {len(to_upload)}")

In [ ]:
BATCH = 40  # tune: 20–80 is fine

def _fmt_bytes(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024 or u == "TB":
            return f"{n:.1f} {u}"
        n /= 1024.0

st = time.time()

# tqdm over batches
for i in tqdm(range(0, len(to_upload), BATCH), desc="Batches", unit="batch"):
    batch = to_upload[i:i+BATCH]
    if not batch:
        continue

    # ----- mirror subfolders inside tmp with a visible progress bar -----
    batch_bytes = sum(p.stat().st_size for p in batch)
    print(f"\n— Batch {i//BATCH+1} | files: {len(batch)} | size: {_fmt_bytes(batch_bytes)}")

    t_copy = time.time()
    for p in tqdm(batch, desc="Copy → local SSD", unit="file"):
        rel = p.relative_to(src)
        dst = tmp/rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)
    copy_sec = time.time() - t_copy
    print(f"Copied in {copy_sec:.1f}s  ({_fmt_bytes(batch_bytes)} @ ~{batch_bytes/1e6/max(copy_sec,1):.1f} MB/s)")

    # ----- upload this batch folder (single commit) -----
    t_up = time.time()
    api.upload_folder(
        folder_path=str(tmp),
        repo_id=repo_id,
        repo_type="dataset",
        path_in_repo="tars",                       # keeps the relative layout
        commit_message=f"Upload batch {i//BATCH+1}",
        ignore_patterns=["*.tar.part", "*.bad", ".*"]
    )
    up_sec = time.time() - t_up
    print(f"Uploaded in {up_sec:.1f}s  ({_fmt_bytes(batch_bytes)} @ ~{batch_bytes/1e6/max(up_sec,1):.1f} MB/s)")

    # ----- clean tmp (ensure we don't accumulate) -----
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)

    # # keep your original "test only first batch" behavior
    # if i == 0:
    #     break

print(f"\nTotal wall time: {time.time()-st:.2f}s")
print("All batches uploaded ✅ (stopped after first batch due to break)" if len(to_upload) > BATCH else "All batches uploaded ✅")

Batches:   0%|          | 0/5 [00:00<?, ?batch/s]


— Batch 1 | files: 40 | size: 68.2 GB


Copy → local SSD:   0%|          | 0/40 [00:00<?, ?file/s]

Copied in 1252.8s  (68.2 GB @ ~58.4 MB/s)


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

...

  /content/tmp/hf_up/baby/baby-00000.tar:   0%|          |  328kB /  187MB            

  /content/tmp/hf_up/boys/boys-00006.tar:   0%|          |  921kB / 2.72GB            

  /content/tmp/hf_up/baby/baby-00002.tar:   0%|          |  253kB /  210MB            

  /content/tmp/hf_up/baby/baby-00001.tar:   0%|          | 1.01MB /  207MB            

Uploaded in 258.4s  (18.9 GB @ ~78.3 MB/s)

Total wall time: 9355.60s
All batches uploaded ✅ (stopped after first batch due to break)


## Recommendation part


### Using Resnet

#### Using Brute Force

In [ ]:
# Local project paths (only used for output + manifest total)
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
SHARDS_STORE    = f"{BASE_DIR}/shards_store"            # embeddings output root
MANIFEST_PATH   = f"{BASE_DIR}/images_manifest.csv.gz"  # used only for tqdm totals

Path(SHARDS_STORE).mkdir(parents=True, exist_ok=True)

# Hugging Face dataset repo (public)
HF_REPO_ID = "PirateKing0402/Amazon-fashion-image-tars"  # <-- change to your repo id

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
# Use list_repo_files instead of list_repo_tree
from huggingface_hub import list_repo_files, hf_hub_url
from typing import List, Dict, Tuple
import os
import webdataset as wds
from PIL import Image

def _hf_tar_urls_for_shard(repo_id: str, shard_name: str, root_prefix: str = "tars/") -> List[str]:
    """List HTTPS URLs to all .tar files for a shard on Hugging Face (robust across HF Hub versions)."""
    files = list_repo_files(repo_id, repo_type="dataset")
    prefix = f"{root_prefix}{shard_name}/"
    urls = [
        hf_hub_url(repo_id, f, repo_type="dataset")
        for f in files
        if f.startswith(prefix) and f.endswith(".tar")
    ]
    urls.sort()
    return urls

def list_shards(_root_ignored: str) -> List[str]:
    """Return shard folder names under tars/ in the HF repo (keeps original signature)."""
    files = list_repo_files(HF_REPO_ID, repo_type="dataset")
    shards = set()
    for f in files:
        # Expect paths like: tars/<shard_name>/<tar_file>.tar
        if f.startswith("tars/") and f.endswith(".tar"):
            parts = f.split("/")
            if len(parts) >= 3:
                shards.add(parts[1])
    return sorted(shards)

def make_wds_for_shard(shard_dir: str):
    """
    Build a WebDataset from HF tar URLs for this shard.
    'shard_dir' is only used to extract shard name (keeps original signature).
    Yields (asin:str, PIL.Image.Image).
    """
    shard_name = os.path.basename(shard_dir.rstrip("/"))
    tar_urls = _hf_tar_urls_for_shard(HF_REPO_ID, shard_name)
    if not tar_urls:
        raise FileNotFoundError(f"No .tar files on HF for shard '{shard_name}'")

    ds = (wds.WebDataset(tar_urls)
          .select(lambda s: ("jpg" in s) or ("png" in s))
          .decode("pil"))

    def mapper(sample: Dict[str, Any]) -> Tuple[str, Image.Image]:
        key  = sample["__key__"]                 # e.g., 'ab/ASIN_hash'
        base = key.split("/")[-1]
        asin = base.split("_")[0]
        img  = (sample["jpg"] if "jpg" in sample else sample["png"]).convert("RGB")
        return asin, img

    return ds.map(mapper)


In [ ]:
IMG_TF = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),  # ImageNet / ResNet
])

def collate_batch(samples):
    ids  = [s[0] for s in samples]
    imgs = [IMG_TF(s[1]) for s in samples]
    return ids, torch.stack(imgs, dim=0)


In [ ]:
EMB_DIM = 2048
EMB_SCHEMA = pa.schema([
    pa.field("row_idx", pa.int64()),
    pa.field("asin",    pa.string()),
    pa.field("shard",   pa.string()),
    pa.field("emb",     pa.list_(pa.float32(), EMB_DIM)), # Corrected: use pa.list_
])

def np2fixed_list_2d(x2d: np.ndarray) -> pa.FixedSizeListArray:
    assert x2d.dtype == np.float32 and x2d.ndim == 2 and x2d.shape[1] == EMB_DIM
    flat = pa.array(x2d.reshape(-1), type=pa.float32())
    return pa.FixedSizeListArray.from_arrays(flat, EMB_DIM)

In [ ]:
def expected_images_for_shard_from_manifest(manifest_gz: str, shard_name: str) -> int:
    """
    Streams the CSV.GZ manifest and counts rows with ok=1 and this shard.
    Gives tqdm a stable total without scanning tar files.
    """
    total = 0
    with gzip.open(manifest_gz, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") == "1" and row.get("shard") == shard_name:
                total += 1
    return total


In [ ]:
@torch.inference_mode()
def embed_shard_to_parquet(shard_name: str, batch_size=128, num_workers=4):
    # NOTE: 'shard_tar_dir' is only used to pass the shard_name into make_wds_for_shard
    shard_tar_dir = f"/virtual/{shard_name}"  # dummy path; make_wds_for_shard extracts basename
    out_dir       = os.path.join(SHARDS_STORE, shard_name)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    out_parquet   = os.path.join(out_dir, "embeddings_raw.parquet")

    # model
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Identity()
    model.eval().to(DEVICE)
    try:
        model = torch.compile(model)  # PyTorch 2.x
    except Exception:
        pass

    # data (now streaming from HF via make_wds_for_shard)
    ds = make_wds_for_shard(shard_tar_dir)
    dl = torch.utils.data.DataLoader(
        ds, batch_size=batch_size, num_workers=num_workers,
        pin_memory=True, collate_fn=collate_batch
    )

    # tqdm total from manifest (if available)
    try:
        total_imgs = expected_images_for_shard_from_manifest(MANIFEST_PATH, shard_name)
    except FileNotFoundError:
        total_imgs = None

    writer = pq.ParquetWriter(out_parquet, EMB_SCHEMA, compression="zstd", write_statistics=False)

    row_idx = 0
    seen = 0
    t0 = time.time()
    try:
        pbar = tqdm(dl, total=None, desc=f"Embedding → {shard_name}") if total_imgs is None \
               else tqdm(dl, total=(total_imgs + batch_size - 1)//batch_size, desc=f"Embedding → {shard_name} (batches)")

        batch = 1
        for ids, imgs in pbar:
            imgs = imgs.to(DEVICE, non_blocking=True)
            feats = model(imgs)                              # [B, 2048]
            feats = nn.functional.normalize(feats, dim=1)    # cosine-ready
            embs  = feats.detach().cpu().numpy().astype(np.float32)

            B = embs.shape[0]
            arr_row   = pa.array(np.arange(row_idx, row_idx+B, dtype=np.int64))
            arr_asin  = pa.array(ids, type=pa.string())
            arr_shard = pa.array([shard_name]*B, type=pa.string())
            arr_emb   = np2fixed_list_2d(embs)

            tbl = pa.Table.from_arrays([arr_row, arr_asin, arr_shard, arr_emb], schema=EMB_SCHEMA)
            writer.write_table(tbl)  # one row-group per batch

            row_idx += B
            seen    += B
            if total_imgs is not None:
                pbar.set_postfix_str(f"{min(seen,total_imgs)}/{total_imgs} imgs")
            if batch == 90:
                break
            batch += 1
    finally:
        writer.close()
    t1 = time.time()

    # Summary + throughput
    elapsed = t1 - t0
    ips = seen / elapsed if elapsed > 0 else 0
    print(f"[{shard_name}] wrote {out_parquet}")
    print(f"Processed {seen} images in {elapsed:.1f}s → {ips:.2f} imgs/s")
    if total_imgs is not None and seen != total_imgs:
        print(f"Note: manifest total={total_imgs}, processed={seen}.")


In [ ]:
shards = list_shards("/ignored")  # argument is ignored by our override
print("HF shards:", shards[:10], "… total:", len(shards))
SHARD = shards[0] if shards else None
SHARD


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


HF shards: ['baby', 'boys', 'clothing|mens', 'clothing|womens', 'girls', 'jewelry|mens', 'jewelry|womens', 'shoes|mens', 'shoes|womens', 'unisex-adult'] … total: 11


'baby'

In [ ]:
if SHARD:
    embed_shard_to_parquet(SHARD, batch_size=128, num_workers=4)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 232MB/s]
/usr/local/lib/python3.12/dist-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


Embedding → baby (batches):   0%|          | 0/360 [00:00<?, ?it/s]

[baby] wrote /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/baby/embeddings_raw.parquet
Processed 11408 images in 96.7s → 118.02 imgs/s
Note: manifest total=46015, processed=11408.


#### Better Feature Extraction

##### Extracting features

In [ ]:
# Local project (outputs + manifest total)
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
SHARDS_STORE    = f"{BASE_DIR}/shards_store"                 # output root
MANIFEST_DL_CSV = f"{BASE_DIR}/images_manifest.csv.gz"       # used only for tqdm totals

Path(SHARDS_STORE).mkdir(parents=True, exist_ok=True)

# Hugging Face (public dataset repo)
HF_REPO_ID = "PirateKing0402/Amazon-fashion-image-tars"  # <-- set this

# Runtime
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 8)//2))   # 4–12 is typical on Colab
BATCH_SIZE  = 128
ROWS_PER_WRITE = 8192          # accumulate then write a parquet part
PARQUET_COMPRESSION = "snappy" # or None for max speed
TAR_BATCH_SIZE = 64            # stream only this many TAR URLs at a time

print("Device:", DEVICE, "| num_workers:", NUM_WORKERS)


Device: cuda | num_workers: 6


In [ ]:
def list_shards(_root_ignored: str) -> List[str]:
    """Return shard folder names under tars/ in the HF repo (keeps original signature)."""
    files = list_repo_files(HF_REPO_ID, repo_type="dataset")
    shards = set()
    for f in files:
        # Expect paths like: tars/<shard>/<file>.tar
        if f.startswith("tars/") and f.endswith(".tar"):
            parts = f.split("/")
            if len(parts) >= 3:
                shards.add(parts[1])
    return sorted(shards)

def hf_tar_url_batches(repo_id: str, shard_name: str, batch_size: int = TAR_BATCH_SIZE, root_prefix: str = "tars/"):
    """Yield small lists of HTTPS TAR URLs for tars/<shard_name>/*.tar (caps RAM)."""
    files = list_repo_files(repo_id, repo_type="dataset")
    tars = [f for f in files if f.startswith(f"{root_prefix}{shard_name}/") and f.endswith(".tar")]
    tars.sort()
    for i in range(0, len(tars), batch_size):
        batch = tars[i:i+batch_size]
        yield [hf_hub_url(repo_id, f, repo_type="dataset") for f in batch]


In [ ]:
def make_wds_for_shard_batch(tar_urls: List[str]):
    """
    Build a WebDataset over a small list of TAR URLs (HF).
    Yields (key:str, asin:str, PIL.Image).
    """
    ds = (wds.WebDataset(tar_urls)
          .select(lambda s: (("jpg" in s) or ("png" in s)))
          .decode("pil"))

    def mapper(sample: Dict[str, Any]) -> Tuple[str, str, Image.Image]:
        key  = sample["__key__"]                 # e.g., 'ab/ASIN_hash'
        base = key.split("/")[-1]
        asin = base.split("_")[0]
        img  = (sample["jpg"] if "jpg" in sample else sample["png"]).convert("RGB")
        return key, asin, img

    return ds.map(mapper)

IMG_TF = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),  # ImageNet / ResNet
])

def collate_batch(samples):
    keys  = [s[0] for s in samples]
    ids   = [s[1] for s in samples]
    imgs  = [IMG_TF(s[2]) for s in samples]
    return keys, ids, torch.stack(imgs, dim=0)


In [ ]:
EMB_DIM = 2048

EMB_SCHEMA = pa.schema([
    pa.field("row_idx", pa.int64()),
    pa.field("__key__", pa.string()),
    pa.field("asin",    pa.string()),
    pa.field("shard",   pa.string()),
    pa.field("emb",     pa.list_(pa.float32(), EMB_DIM)),
])

MANIFEST_SCHEMA = pa.schema([
    pa.field("row_idx", pa.int64()),
    pa.field("__key__", pa.string()),
    pa.field("asin",    pa.string()),
    pa.field("shard",   pa.string()),
    pa.field("ok",      pa.int8()),    # 1 ok, 0 error
    pa.field("error",   pa.string()),
    pa.field("part",    pa.string()),  # parquet part path
    pa.field("ts_ms",   pa.int64()),
])

def np2fixed_list_2d(x2d: np.ndarray) -> pa.FixedSizeListArray:
    assert x2d.dtype == np.float32 and x2d.ndim == 2 and x2d.shape[1] == EMB_DIM
    flat = pa.array(x2d.reshape(-1), type=pa.float32())
    return pa.FixedSizeListArray.from_arrays(flat, EMB_DIM)


In [ ]:
def expected_images_for_shard_from_manifest(manifest_gz: str, shard_name: str) -> int:
    total = 0
    with gzip.open(manifest_gz, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") == "1" and row.get("shard") == shard_name:
                total += 1
    return total


In [ ]:
# NOTE:
# - WAL for concurrency and speed.
# - synchronous=OFF is the FASTEST but can lose very recent writes on a sudden crash/power-loss.
#   You requested OFF; if you want safer middle ground, change to NORMAL.

def _db_paths(shard_name: str):
    out_dir = os.path.join(SHARDS_STORE, shard_name)
    emb_dir = os.path.join(out_dir, "emb_parts")
    man_dir = os.path.join(out_dir, "emb_manifest")
    Path(emb_dir).mkdir(parents=True, exist_ok=True)
    Path(man_dir).mkdir(parents=True, exist_ok=True)
    return out_dir, emb_dir, man_dir, os.path.join(out_dir, "processed.sqlite")

def open_db(shard_name: str):
    out_dir, emb_dir, man_dir, db_path = _db_paths(shard_name)
    conn = sqlite3.connect(db_path, isolation_level=None, check_same_thread=False)
    cur  = conn.cursor()
    cur.execute("PRAGMA journal_mode=WAL")
    cur.execute("PRAGMA synchronous=OFF")  # <- as requested (fastest)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS processed (
            __key__ TEXT PRIMARY KEY,
            status  TEXT CHECK(status IN ('pending','ok')) NOT NULL
        )
    """)
    return conn

def cleanup_stale_pending(conn):
    cur = conn.cursor()
    cur.execute("DELETE FROM processed WHERE status='pending'")

def count_ok(conn) -> int:
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM processed WHERE status='ok'")
    return int(cur.fetchone()[0])

def seed_ok_from_existing_embeddings(conn, shard_name: str):
    """
    One quick pass to ensure DB matches any already-written embeddings.
    Prevents duplicates if a past run wrote vectors but crashed before DB update.
    """
    out_dir, emb_dir, man_dir, _ = _db_paths(shard_name)
    part_glob = os.path.join(emb_dir, "part-*.parquet")
    parts = sorted(glob.glob(part_glob))
    if not parts:
        # fallback single file (if you had it earlier)
        return

    cur = conn.cursor()
    for fp in parts:
        pf = pq.ParquetFile(fp)
        cols = pf.schema_arrow.names
        key_col = "__key__" if "__key__" in cols else None
        if key_col is None:
            # If legacy files lack __key__, skip seeding
            continue
        for rg in range(pf.num_row_groups):
            tbl = pf.read_row_group(rg, columns=[key_col])
            keys = [(str(k),) for k in tbl[key_col].to_pylist()]
            cur.executemany(
                "INSERT OR IGNORE INTO processed(__key__, status) VALUES (?, 'ok')", keys
            )

def reserve_pending(conn, keys, chunk_size=512):
    """
    Try to reserve each key (INSERT OR IGNORE) in small chunks.
    Return indices in `keys` that were actually inserted now (i.e., new work).
    """
    if not keys:
        return []

    keep_idx = []
    cur = conn.cursor()

    for start in range(0, len(keys), chunk_size):
        chunk = keys[start:start + chunk_size]
        for off, k in enumerate(chunk):
            cur.execute(
                "INSERT OR IGNORE INTO processed(__key__, status) VALUES (?, 'pending')",
                (k,)
            )
            # cur.rowcount == 1 => this INSERT created a new row
            # cur.rowcount == 0 => it was ignored (already pending/ok)
            if cur.rowcount == 1:
                keep_idx.append(start + off)

    return keep_idx

def mark_ok(conn, keys: List[str]):
    if not keys:
        return
    cur = conn.cursor()
    CHUNK = 500
    for i in range(0, len(keys), CHUNK):
        chunk = keys[i:i+CHUNK]
        q = ",".join(["?"]*len(chunk))
        cur.execute(f"UPDATE processed SET status='ok' WHERE __key__ IN ({q}) AND status='pending'", chunk)

def release_pending(conn, keys: List[str]):
    if not keys:
        return
    cur = conn.cursor()
    CHUNK = 500
    for i in range(0, len(keys), CHUNK):
        chunk = keys[i:i+CHUNK]
        q = ",".join(["?"]*len(chunk))
        cur.execute(f"DELETE FROM processed WHERE __key__ IN ({q}) AND status='pending'", chunk)


In [ ]:
@torch.inference_mode()
def embed_shard_to_parquet(shard_name: str,
                           batch_size: int = BATCH_SIZE,
                           num_workers: int = NUM_WORKERS,
                           rows_per_write: int = ROWS_PER_WRITE,
                           compression: str = PARQUET_COMPRESSION,
                           tar_batch_size: int = TAR_BATCH_SIZE):
    out_dir, emb_dir, man_dir, _ = _db_paths(shard_name)

    # ---- SQLite uniqueness barrier ----
    conn = open_db(shard_name)
    cleanup_stale_pending(conn)                # clear leftovers from dead runs
    seed_ok_from_existing_embeddings(conn, shard_name)  # align DB with any existing parts
    ok_so_far = count_ok(conn)

    # ---- TQDM total from download manifest (minus already ok) ----
    try:
        exp_total = expected_images_for_shard_from_manifest(MANIFEST_DL_CSV, shard_name)
    except FileNotFoundError:
        exp_total = None
    remaining = None if exp_total is None else max(0, exp_total - ok_so_far)
    print(f"[{shard_name}] remaining={remaining}")
    print(f"[{shard_name}] total={exp_total}")
    print(f"[{shard_name}] ok_so_far={ok_so_far}")
    pbar = tqdm(total=remaining, desc=f"Embedding → {shard_name}", unit="img")

    # ---- Model ----
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Identity()
    try:
        model = torch.compile(model)
    except Exception:
        pass
    model.eval().to(DEVICE)

    def to_device(x: torch.Tensor) -> torch.Tensor:
        if DEVICE == "cuda":
            return x.to(memory_format=torch.channels_last).to(DEVICE, non_blocking=True)
        return x.to(DEVICE, non_blocking=True)

    # ---- Buffers (write parts after rows_per_write) ----
    run_id   = f"{time.strftime('%Y%m%d-%H%M%S')}-{random.randint(1000,9999)}"
    part_seq = 0
    next_row = ok_so_far

    buf_keys, buf_asins, buf_embs, buf_rows = [], [], [], 0

    def flush_buffers():
        nonlocal buf_keys, buf_asins, buf_embs, buf_rows, part_seq, next_row
        if buf_rows == 0:
            return
        embs = np.vstack(buf_embs).astype(np.float32, copy=False)
        arr_row   = pa.array(np.arange(next_row, next_row + buf_rows, dtype=np.int64))
        arr_key   = pa.array(buf_keys,  type=pa.string())
        arr_asin  = pa.array(buf_asins, type=pa.string())
        arr_shard = pa.array([shard_name]*buf_rows, type=pa.string())
        arr_emb   = np2fixed_list_2d(embs)

        part_path = os.path.join(emb_dir, f"part-{run_id}-{part_seq:05d}.parquet")
        emb_tbl = pa.Table.from_arrays([arr_row, arr_key, arr_asin, arr_shard, arr_emb], schema=EMB_SCHEMA)
        pq.write_table(emb_tbl, part_path, compression=compression, write_statistics=False)

        ts = int(time.time()*1000)
        man_tbl = pa.Table.from_arrays([
            arr_row, arr_key, arr_asin, arr_shard,
            pa.array([1]*buf_rows, type=pa.int8()),
            pa.array([""]*buf_rows, type=pa.string()),
            pa.array([part_path]*buf_rows, type=pa.string()),
            pa.array([ts]*buf_rows, type=pa.int64())
        ], schema=MANIFEST_SCHEMA)
        pq.write_table(man_tbl, os.path.join(man_dir, f"manifest-{run_id}-{part_seq:05d}.parquet"),
                       compression=compression, write_statistics=False)

        next_row += buf_rows
        part_seq += 1
        buf_keys.clear(); buf_asins.clear(); buf_embs.clear(); buf_rows = 0

    seen_written = 0
    t0 = time.time()
    try:
        # ---- Iterate small TAR URL batches to cap RAM ----
        for tar_urls in hf_tar_url_batches(HF_REPO_ID, shard_name, batch_size=tar_batch_size):
            ds = make_wds_for_shard_batch(tar_urls)

            dl_kwargs = dict(batch_size=batch_size, num_workers=num_workers,
                             pin_memory=True, collate_fn=collate_batch)
            if num_workers > 0:
                dl_kwargs.update(persistent_workers=True, prefetch_factor=4)
            dl = torch.utils.data.DataLoader(ds, **dl_kwargs)

            for keys, asins, imgs in dl:
                # 1) Reserve (exact uniqueness). Keep only newly-reserved keys.
                hold_idx = reserve_pending(conn, keys)
                if not hold_idx:
                    continue
                if len(hold_idx) != len(keys):
                    sel = hold_idx
                    keys  = [keys[i]  for i in sel]
                    asins = [asins[i] for i in sel]
                    imgs  = imgs[sel]

                # 2) Run model
                imgs = to_device(imgs)
                feats = model(imgs)
                feats = nn.functional.normalize(feats, dim=1)
                embs  = feats.detach().cpu().numpy().astype(np.float32)

                # 3) Buffer → maybe flush
                B = embs.shape[0]
                buf_keys.extend(keys)
                buf_asins.extend(asins)
                buf_embs.append(embs)
                buf_rows += B
                seen_written += B
                pbar.update(B)

                if buf_rows >= rows_per_write:
                    flush_buffers()
                    # 4) Mark OK for keys just flushed
                    mark_ok(conn, buf_keys)  # NOTE: buf cleared right after flush; mark just-written

            # cleanup per TAR batch
            del dl, ds
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    except (Exception, KeyboardInterrupt) as e:
        release_pending(conn, buf_keys)
        print("Successfully deleted all pending items to avoid clash.")
        if isinstance(e, KeyboardInterrupt):
            print("Interrupted by user.")
            raise
    finally:
        # final flush & mark
        if buf_rows > 0:
            flush_buffers()
            mark_ok(conn, buf_keys)
        pbar.close()
        conn.close()

    elapsed = time.time() - t0
    ips = seen_written / max(elapsed, 1e-9)
    print(f"[{shard_name}] DONE. Wrote parts in {emb_dir} and manifests in {man_dir}.")
    print(f"Processed {seen_written} images in {elapsed:.1f}s → {ips:.2f} imgs/s")


In [ ]:
shards = list_shards("/ignored")
print("HF shards:", shards[:10], "… total:", len(shards))
SHARD = shards[0] if shards else None
SHARD


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


HF shards: ['baby', 'boys', 'clothing|mens', 'clothing|womens', 'girls', 'jewelry|mens', 'jewelry|womens', 'shoes|mens', 'shoes|womens', 'unisex-adult'] … total: 11


'baby'

In [ ]:
for SHARD in shards:
    embed_shard_to_parquet(
        shard_name=SHARD,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        rows_per_write=ROWS_PER_WRITE,
        compression=PARQUET_COMPRESSION,
        tar_batch_size=TAR_BATCH_SIZE
    )


[baby] remaining=0
[baby] total=46015
[baby] ok_so_far=46015


Embedding → baby: 0img [00:00, ?img/s]

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 15%|█▌        | 15.1M/97.8M [00:00<00:00, 158MB/s]
 35%|███▍      | 33.8M/97.8M [00:00<00:00, 180MB/s]
 57%|█████▋    | 56.1M/97.8M [00:00<00:00, 204MB/s]
 77%|███████▋  | 75.8M/97.8M [00:00<00:00, 199MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]
/usr/local/lib/python3.12/dist-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


[baby] DONE. Wrote parts in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/baby/emb_parts and manifests in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/baby/emb_manifest.
Processed 0 images in 325.9s → 0.00 imgs/s
[boys] remaining=0
[boys] total=70068
[boys] ok_so_far=70068


...

[shoes|womens] DONE. Wrote parts in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/shoes|womens/emb_parts and manifests in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/shoes|womens/emb_manifest.
Processed 471595 images in 2249.2s → 209.68 imgs/s
[unisex-adult] remaining=33238
[unisex-adult] total=49694
[unisex-adult] ok_so_far=16456


Embedding → unisex-adult:   0%|          | 0/33238 [00:00<?, ?img/s]

[unisex-adult] DONE. Wrote parts in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/unisex-adult/emb_parts and manifests in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/unisex-adult/emb_manifest.
Processed 33238 images in 300.0s → 110.78 imgs/s
[unisex-child] remaining=22612
[unisex-child] total=38998
[unisex-child] ok_so_far=16386


Embedding → unisex-child:   0%|          | 0/22612 [00:00<?, ?img/s]

[unisex-child] DONE. Wrote parts in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/unisex-child/emb_parts and manifests in /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/shards_store/unisex-child/emb_manifest.
Processed 22612 images in 295.4s → 76.55 imgs/s


##### Checking manifest files of embeddings generated

In [ ]:
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
SHARDS_STORE    = f"{BASE_DIR}/shards_store"
DOWN_MANIFEST   = f"{BASE_DIR}/images_manifest.csv.gz"   # original download manifest with 'ok' and 'shard'
HF_REPO_ID = "PirateKing0402/Amazon-fashion-image-tars"

In [ ]:
def list_shards(_root_ignored: str) -> List[str]:
    """Return shard folder names under tars/ in the HF repo (keeps original signature)."""
    files = list_repo_files(HF_REPO_ID, repo_type="dataset")
    shards = set()
    for f in files:
        # Expect paths like: tars/<shard>/<file>.tar
        if f.startswith("tars/") and f.endswith(".tar"):
            parts = f.split("/")
            if len(parts) >= 3:
                shards.add(parts[1])
    return sorted(shards)

In [ ]:
shards = list_shards("/ignored")
print("HF shards:", shards[:10], "… total:", len(shards))
SHARD = shards[0] if shards else None
SHARD

HF shards: ['baby', 'boys', 'clothing|mens', 'clothing|womens', 'girls', 'jewelry|mens', 'jewelry|womens', 'shoes|mens', 'shoes|womens', 'unisex-adult'] … total: 11


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


'baby'

In [ ]:
okks = []
for SHARD in shards:

    man_dir = os.path.join(SHARDS_STORE, SHARD, "emb_manifest")
    emb_dir = os.path.join(SHARDS_STORE, SHARD, "emb_parts")

    print("man_dir exists:", os.path.isdir(man_dir), "| emb_dir exists:", os.path.isdir(emb_dir))

    part_paths = sorted(glob.glob(os.path.join(man_dir, "manifest-*.parquet")))
    print(f"{SHARD}: {len(part_paths)} manifest part files")

    ok = 0
    err = 0
    err_msgs = Counter()
    keys_seen = set()
    dups = 0

    for fp in tqdm(part_paths, desc="Scan manifests"):
        pf = pq.ParquetFile(fp)
        for rg in range(pf.num_row_groups):
            tbl = pf.read_row_group(rg, columns=["__key__","ok","error"])
            k = tbl["__key__"].to_pylist()
            o = tbl["ok"].to_numpy()
            e = tbl["error"].to_pylist()
            for i, kk in enumerate(k):
                if kk in keys_seen:
                    dups += 1
                else:
                    keys_seen.add(kk)
                if o[i] == 1:
                    ok += 1
                else:
                    err += 1
                    err_msgs[e[i]] += 1
    okks.append(ok)

    print(f"\n[{SHARD}] ok={ok:,}  errors={err:,}  duplicate_keys_in_manifest={dups}")
    print("Top error reasons:", err_msgs.most_common(10))


man_dir exists: True | emb_dir exists: True
baby: 6 manifest part files


Scan manifests:   0%|          | 0/6 [00:00<?, ?it/s]


[baby] ok=46,015  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: True | emb_dir exists: True
boys: 9 manifest part files


Scan manifests:   0%|          | 0/9 [00:00<?, ?it/s]


[boys] ok=70,068  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: True | emb_dir exists: True
clothing|mens: 36 manifest part files


...


[shoes|womens] ok=487,979  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: True | emb_dir exists: True
unisex-adult: 7 manifest part files


Scan manifests:   0%|          | 0/7 [00:00<?, ?it/s]


[unisex-adult] ok=49,694  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: True | emb_dir exists: True
unisex-child: 5 manifest part files


Scan manifests:   0%|          | 0/5 [00:00<?, ?it/s]


[unisex-child] ok=38,998  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []


In [ ]:
for i,SHARD in enumerate(shards):
    expected = 0
    with gzip.open(DOWN_MANIFEST, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") == "1" and row.get("shard") == SHARD:
                expected += 1

    print(f"[{SHARD}] expected from download manifest: {expected:,}")
    if expected:
        print(f"coverage: {okks[i]/expected:.2%}")


[clothing|mens] expected from download manifest: 288,057
coverage: 100.00%


##### Finding recommendations

In [5]:
import os
import glob
import shutil
import sqlite3
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pyarrow.parquet as pq
import faiss
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
SHARDS_STORE    = f"{BASE_DIR}/shards_store"
DRIVE_DB_PATH   = f"{BASE_DIR}/asin2url.sqlite"
LOCAL_DB_PATH   = "/content/asin2url.sqlite"

# FAISS Index Files
INDEX_TRAINED_PATH = os.path.join(BASE_DIR, "trained_IVF4096.index")  # The "Skeleton" (Clusters)
IVF_DATA_PATH      = os.path.join(BASE_DIR, "vectors_data.ivfdata")   # The "Body" (Heavy Data on Disk)

# Metadata Storage (SQLite instead of Pickle/RAM)
METADATA_DB_PATH   = os.path.join(BASE_DIR, "faiss_metadata.sqlite")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
IMG_TF = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def get_inference_model():
    """Load ResNet50 (Embedding Extractor)"""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Identity()
    model.eval()
    model.to(DEVICE)
    return model

In [7]:
def init_metadata_db():
    """Creates a fresh SQLite DB to store the FAISS RowID -> ASIN mapping"""
    # Start fresh to guarantee IDs match FAISS exactly (0, 1, 2...)
    if os.path.exists(METADATA_DB_PATH):
        os.remove(METADATA_DB_PATH)

    conn = sqlite3.connect(METADATA_DB_PATH)
    cursor = conn.cursor()

    # Optimization: NORMAL is faster than FULL but safer than OFF
    # It protects against app crashes, just not full power outages (perfect for Colab)
    cursor.execute("PRAGMA synchronous = NORMAL")
    cursor.execute("PRAGMA journal_mode = DELETE")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS index_map (
            row_id INTEGER PRIMARY KEY,
            asin TEXT
        )
    """)
    conn.commit()
    return conn

In [8]:
def build_disk_index_all_shards(shard_dirs):
    if os.path.exists(INDEX_TRAINED_PATH) and os.path.exists(IVF_DATA_PATH) and os.path.exists(METADATA_DB_PATH):
        print(f"Index found at {INDEX_TRAINED_PATH}. Skipping build.")
        return

    print("Building new Disk-Backed Index (1.5M Items)...")

    # A. CONFIGURATION
    d = 2048
    nlist = 4096
    SAMPLING_RATE = 0.10

    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

    # --- HELPER: PRE-CALCULATE TOTAL FILES FOR PROGRESS BAR ---
    all_parquet_files = []
    for shard_name in shard_dirs:
        emb_dir = os.path.join(SHARDS_STORE, shard_name, "emb_parts")
        if os.path.exists(emb_dir):
            all_parquet_files.extend(glob.glob(os.path.join(emb_dir, "*.parquet")))

    # Generator now accepts a 'desc' for the progress bar label
    def generate_all_batches(desc_text):
        # tqdm wraps the list of files to show progress
        for f in tqdm(all_parquet_files, desc=desc_text, unit="file"):
            table = pq.read_table(f)
            embs = np.vstack(table["emb"].to_numpy())
            asins = table["asin"].to_numpy()
            yield embs, asins

    # --- PHASE 1: UNBIASED SAMPLING & TRAINING ---
    print("\n--- Phase 1: Training (Sampling 10%) ---")
    train_vectors = []

    # Pass 1: Scan files to get sample
    for embs, _ in generate_all_batches("Sampling"):
        num_items = len(embs)
        mask = np.random.rand(num_items) < SAMPLING_RATE
        if np.any(mask):
            train_vectors.append(embs[mask])

    if not train_vectors:
        print("Error: No training data found.")
        return

    training_data = np.concatenate(train_vectors, axis=0).astype('float32')
    print(f"Training Clustering on {training_data.shape[0]} vectors...")
    index.train(training_data)
    del training_data, train_vectors
    print("Training complete.")

    # --- PHASE 2: SETUP DISK STORAGE ---
    faiss.write_index(index, INDEX_TRAINED_PATH)
    invlists = faiss.OnDiskInvertedLists(index.nlist, index.code_size, IVF_DATA_PATH)
    index.replace_invlists(invlists)

    # --- PHASE 3: SYNCHRONIZED POPULATION ---
    print("\n--- Phase 3: Populating Index & Metadata ---")

    meta_conn = init_metadata_db()
    meta_cursor = meta_conn.cursor()

    BATCH_SIZE = 50000
    vec_buffer = []
    asin_buffer = []
    global_row_id = 0

    # Pass 2: Scan files to add data
    for embs, asins in generate_all_batches("Indexing"):
        embs_f32 = embs.astype('float32')
        faiss.normalize_L2(embs_f32)

        vec_buffer.append(embs_f32)

        for asin in asins:
            asin_buffer.append((global_row_id, asin))
            global_row_id += 1

        if sum(len(x) for x in vec_buffer) >= BATCH_SIZE:
            try:
                flush_vecs = np.concatenate(vec_buffer, axis=0)
                index.add(flush_vecs)
                meta_cursor.executemany("INSERT INTO index_map (row_id, asin) VALUES (?, ?)", asin_buffer)
                meta_conn.commit()

                vec_buffer = []
                asin_buffer = []
            except Exception as e:
                print(f"Error at ID {global_row_id}: {e}")
                break

    # Final Flush
    if vec_buffer:
        flush_vecs = np.concatenate(vec_buffer, axis=0)
        index.add(flush_vecs)
        meta_cursor.executemany("INSERT INTO index_map (row_id, asin) VALUES (?, ?)", asin_buffer)
        meta_conn.commit()

    meta_conn.close()
    print(f"\nBuild Complete! Total items: {global_row_id}")

In [9]:
def load_disk_index_resources():
    print("Loading Index Header...")
    # 1. Load the Skeleton
    index = faiss.read_index(INDEX_TRAINED_PATH)

    # 2. CRITICAL FIX: Sync the Item Count
    # The header thinks ntotal is 0. We verify with the Database.
    if os.path.exists(METADATA_DB_PATH):
        conn = sqlite3.connect(METADATA_DB_PATH)
        cursor = conn.cursor()
        # Count rows in the mapping table
        cursor.execute("SELECT COUNT(*) FROM index_map")
        real_count = cursor.fetchone()[0]
        conn.close()

        if real_count > 0:
            print(f"⚠️ Header said 0 items. Correcting to: {real_count}")
            index.ntotal = real_count # Force update the count
        else:
            print("Warning: Database is also empty!")

    # 3. Check if Data File exists and is not empty
    if not os.path.exists(IVF_DATA_PATH) or os.path.getsize(IVF_DATA_PATH) == 0:
        raise FileNotFoundError(f"CRITICAL: {IVF_DATA_PATH} is missing or empty (0 bytes).")

    print(f"Attaching Disk Data ({os.path.getsize(IVF_DATA_PATH)/1024**3:.2f} GB)...")
    invlists = faiss.OnDiskInvertedLists(index.nlist, index.code_size, IVF_DATA_PATH)
    index.replace_invlists(invlists)

    index.nprobe = 10
    return index

In [10]:
def get_asin_by_faiss_id(faiss_id):
    """Fetch single ASIN from the metadata DB using Row ID"""
    conn = sqlite3.connect(METADATA_DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT asin FROM index_map WHERE row_id=?", (int(faiss_id),))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else None

In [11]:
def find_similar_products_faiss(model, query_img_path, index, top_k=5):
    # A. Preprocess
    img = Image.open(query_img_path).convert("RGB")
    img_tensor = IMG_TF(img).unsqueeze(0).to(DEVICE)

    # B. Extract & Normalize
    with torch.no_grad():
        query_emb = model(img_tensor).cpu().numpy().astype('float32')
        faiss.normalize_L2(query_emb)

    # C. Search (Disk I/O happens here automatically via MMAP)
    D, I = index.search(query_emb, top_k)

    # D. Map IDs to ASINs (SQL Lookup)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx != -1:
            asin = get_asin_by_faiss_id(idx)
            if asin:
                results.append((asin, score))

    return results

In [12]:
def get_urls_from_sqlite(asin_list, db_path=LOCAL_DB_PATH):
    if not asin_list: return {}
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    placeholders = ',' .join('?' for _ in asin_list)
    query = f"SELECT parent_asin, url FROM asin2url WHERE parent_asin IN ({placeholders})"
    cursor.execute(query, asin_list)
    results = cursor.fetchall()
    conn.close()
    return {row[0]: row[1] for row in results}

In [13]:
def show_results_with_url(query_path, results, url_map):
    plt.figure(figsize=(15, 5))

    # Query
    ax = plt.subplot(1, len(results) + 1, 1)
    ax.imshow(Image.open(query_path))
    ax.set_title("Query")
    ax.axis('off')

    # Matches
    for i, (asin, score) in enumerate(results):
        ax = plt.subplot(1, len(results) + 1, i + 2)
        url = url_map.get(asin)
        if url:
            try:
                resp = requests.get(url, stream=True, timeout=2)
                if resp.status_code == 200:
                    ax.imshow(Image.open(resp.raw))
                else: ax.text(0.5,0.5,"Img Fail",ha='center')
            except: ax.text(0.5,0.5,"Err",ha='center')
        else: ax.text(0.5,0.5,"No URL",ha='center')
        ax.set_title(f"{asin}\n{score:.3f}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [14]:
# Setup URL DB (existing helper)
if not os.path.exists(LOCAL_DB_PATH) and os.path.exists(DRIVE_DB_PATH):
    print("Copying URL DB locally...")
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB_PATH)

In [15]:
# --- MAIN FLOW ---
model = get_inference_model()
all_shards = sorted(os.listdir(SHARDS_STORE))
print(f"Found {len(all_shards)} shards.")

if all_shards:
    # 1. Build Index (Safe, Fast, Fault Tolerant)
    build_disk_index_all_shards(all_shards)

    # 2. Load Index Header
    disk_index = load_disk_index_resources()
    print(f"System Ready. Total Items in Index: {disk_index.ntotal}")
else:
    print("Error: No shards found.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]


Found 11 shards.
Building new Disk-Backed Index (1.5M Items)...

--- Phase 1: Training (Sampling 10%) ---


Sampling: 100%|██████████| 255/255 [19:12<00:00,  4.52s/file]


Training Clustering on 202774 vectors...
Training complete.

--- Phase 3: Populating Index & Metadata ---


Indexing: 100%|██████████| 255/255 [2:08:58<00:00, 30.35s/file]



Build Complete! Total items: 2032692
Loading Index Header...
Attaching Disk Data from /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/vectors_data.ivfdata...
System Ready. Total Items in Index: 0


In [ ]:
# --- MAIN FLOW (INFERENCE ONLY) ---

# 1. Mount Drive to access existing Index & Metadata
from google.colab import drive
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# 2. Setup URL DB (Copy from Drive to Local Colab for speed)
if not os.path.exists(LOCAL_DB_PATH) and os.path.exists(DRIVE_DB_PATH):
    print("Copying URL DB locally for faster lookups...")
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB_PATH)

# 3. Load Resources (Skip Building)
# We use the paths defined at the top of your script (INDEX_TRAINED_PATH, etc.)
if os.path.exists(INDEX_TRAINED_PATH) and os.path.exists(IVF_DATA_PATH):
    print("Found existing index in Drive. Loading...")

    # Load the Index
    disk_index = load_disk_index_resources()
    print(f"System Ready. Total Items in Index: {disk_index.ntotal}")

    # Load the Model
    model = get_inference_model()

    # 4. Run Test Search
    from google.colab import files
    print("\nUpload a query image...")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f"Searching for {filename}...")
        try:
            matches = find_similar_products_faiss(model, filename, disk_index, top_k=5)

            if matches:
                top_asins = [m[0] for m in matches]
                urls = get_urls_from_sqlite(top_asins)
                show_results_with_url(filename, matches, urls)
            else:
                print("No matches found.")
        except Exception as e:
            print(f"Error during search: {e}")

else:
    print(f"CRITICAL ERROR: Index files not found in {BASE_DIR}")
    print("Please check if 'trained_IVF4096.index' and 'vectors_data.ivfdata' exist in that folder.")

Found existing index in Drive. Loading...
Loading Index Header...
⚠️ Header said 0 items. Correcting to: 2032692
Attaching Disk Data (64.00 GB)...
System Ready. Total Items in Index: 2032692

Upload a query image...


Saving dress.jpg to dress (4).jpg
Searching for dress (4).jpg...
